# 02_silver_simar.ipynb — Limpieza SIMAR Bronze → Silver

Este notebook procesa los CSV de **Puertos del Estado / SIMAR** para generar tablas Silver:

- `silver/ocean_hourly/` desde archivos `WAVE`
- `silver/meteo_hourly/` desde archivos `WIND`
- `silver/ocean_physics/` desde archivos `CURRENTS`, `WATER_TEMP` y `SALINITY`

Requiere que ya exista:

```text
silver/beach_geography/beach_geography.parquet
```

La tabla `beach_geography` se usa como `dim_zone` para asignar cada punto SIMAR a la zona costera/playa más cercana.

**Versión corregida:** añade validación estricta de timestamps para evitar fechas falsas en 1970.

## Celda 0 — Montar Google Drive

In [40]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Celda 1 — Instalar librerías necesarias

In [41]:
!pip -q install geopandas pyarrow shapely fiona tqdm

## Celda 2 — Imports, rutas y configuración

In [42]:
from pathlib import Path
import pandas as pd
import numpy as np
import geopandas as gpd
import re
import unicodedata
import json
import shutil
from functools import reduce
from tqdm.auto import tqdm

BASE_DIR = Path("/content/drive/MyDrive/AI Projects/DeepWave Canarias")
BRONZE_DIR = BASE_DIR / "data" / "bronze"

SILVER_DIR = BASE_DIR / "silver"

SIMAR_DIR = BRONZE_DIR / "Puertos del Estado" / "SIMAR"
DIM_ZONE_PATH = SILVER_DIR / "beach_geography" / "beach_geography.parquet"

OUT_OCEAN_DIR = SILVER_DIR / "ocean_hourly"
OUT_METEO_DIR = SILVER_DIR / "meteo_hourly"
OUT_PHYSICS_DIR = SILVER_DIR / "ocean_physics"

QC_DIR = SILVER_DIR / "_quality_reports"
META_DIR = SILVER_DIR / "_metadata"

QC_DIR.mkdir(parents=True, exist_ok=True)
META_DIR.mkdir(parents=True, exist_ok=True)
OUT_OCEAN_DIR.mkdir(parents=True, exist_ok=True)
OUT_METEO_DIR.mkdir(parents=True, exist_ok=True)
OUT_PHYSICS_DIR.mkdir(parents=True, exist_ok=True)

BBOX_CANARIAS = {
    "lat_min": 27.0,
    "lat_max": 29.5,
    "lon_min": -18.5,
    "lon_max": -13.0,
}

SOURCE_NAME = "SIMAR"

print("BASE_DIR:", BASE_DIR)
print("BRONZE_DIR existe:", BRONZE_DIR.exists())
print("SIMAR_DIR existe:", SIMAR_DIR.exists())
print("DIM_ZONE_PATH existe:", DIM_ZONE_PATH.exists())

if not BRONZE_DIR.exists():
    raise FileNotFoundError("No existe BRONZE_DIR. Revisa que Drive esté montado.")

if not SIMAR_DIR.exists():
    raise FileNotFoundError(f"No existe SIMAR_DIR: {SIMAR_DIR}")

if not DIM_ZONE_PATH.exists():
    raise FileNotFoundError(
        "No existe beach_geography.parquet. Ejecuta primero 01_silver_dim_zone.ipynb."
    )

BASE_DIR: /content/drive/MyDrive/AI Projects/DeepWave Canarias
BRONZE_DIR existe: True
SIMAR_DIR existe: True
DIM_ZONE_PATH existe: True


## Celda 3 — Utilidades generales

In [43]:
def normalize_text(value):
    if pd.isna(value):
        return np.nan

    value = str(value).strip()
    value = unicodedata.normalize("NFKD", value)
    value = "".join(c for c in value if not unicodedata.combining(c))
    value = re.sub(r"\s+", " ", value)

    return value.upper()


def slugify(value):
    value = normalize_text(value)

    if pd.isna(value):
        return "UNKNOWN"

    value = re.sub(r"[^A-Z0-9]+", "_", value)
    value = re.sub(r"_+", "_", value)

    return value.strip("_")


def normalize_col(col):
    col = normalize_text(col)
    if pd.isna(col):
        return ""
    col = re.sub(r"[^A-Z0-9]+", "_", col)
    col = re.sub(r"_+", "_", col).strip("_")
    return col


def parse_coordinate(value):
    """Convierte coordenadas decimal/DMS a float; soporta N/S/E/W/O."""
    if pd.isna(value):
        return np.nan

    if isinstance(value, (int, float, np.integer, np.floating)):
        return float(value)

    s = str(value).strip().upper()
    s = s.replace(",", ".")

    if s in ["", "NAN", "NONE", "NULL"]:
        return np.nan

    sign = 1
    if any(h in s for h in ["W", "O", "S"]):
        sign = -1
    if s.startswith("-"):
        sign = -1

    nums = re.findall(r"-?\d+(?:\.\d+)?", s)

    if not nums:
        return np.nan

    try:
        if len(nums) >= 3 and ("º" in s or "°" in s or "'" in s or '"' in s):
            deg = abs(float(nums[0]))
            minutes = float(nums[1])
            seconds = float(nums[2])
            value = deg + minutes / 60 + seconds / 3600

        elif len(nums) >= 2 and ("º" in s or "°" in s or "'" in s):
            deg = abs(float(nums[0]))
            minutes = float(nums[1])
            value = deg + minutes / 60

        else:
            value = abs(float(nums[0])) if sign == -1 else float(nums[0])

        return sign * abs(value) if sign == -1 else value

    except Exception:
        return np.nan


def to_numeric_series(series):
    """Convierte columnas numéricas con coma decimal, unidades o códigos de missing."""
    if series is None:
        return pd.Series(dtype="float64")

    s = series.astype(str).str.strip()

    missing_tokens = {
        "",
        "NA",
        "N/A",
        "NAN",
        "NULL",
        "NONE",
        "-",
        "--",
        "---",
        "S/D",
        "SD",
    }

    s = s.mask(s.str.upper().isin(missing_tokens))
    s = s.str.replace(",", ".", regex=False)
    s = s.str.replace(r"[^0-9eE+\-.]", "", regex=True)

    out = pd.to_numeric(s, errors="coerce")

    # Códigos habituales de dato ausente en series oceanográficas.
    out = out.mask(out.isin([-99999, -9999, -999, 999, 9999, 99999]))

    return out


def infer_column(df, candidates):
    cols_norm = {normalize_col(col): col for col in df.columns}

    for candidate in candidates:
        candidate_norm = normalize_col(candidate)

        for col_norm, original_col in cols_norm.items():
            if candidate_norm == col_norm:
                return original_col

        for col_norm, original_col in cols_norm.items():
            if candidate_norm in col_norm:
                return original_col

    return None


def find_col_by_patterns(df, patterns, exclude_patterns=None):
    """Busca una columna por patrones regex aplicados sobre nombres normalizados."""
    if exclude_patterns is None:
        exclude_patterns = []

    for col in df.columns:
        norm = normalize_col(col)

        if any(re.search(ex, norm) for ex in exclude_patterns):
            continue

        if any(re.search(pat, norm) for pat in patterns):
            return col

    return None


def looks_like_date_string(value):
    s = str(value)
    return bool(
        re.search(r"\d{1,2}[/-]\d{1,2}[/-]\d{2,4}", s)
        or re.search(r"\d{4}[/-]\d{1,2}[/-]\d{1,2}", s)
    )


def safe_year_from_timestamp(ts):
    return pd.to_datetime(ts, utc=True, errors="coerce").dt.year.astype("Int64")

## Celda 4 — Cargar `dim_zone` / `beach_geography`

In [44]:
beach_geography = pd.read_parquet(DIM_ZONE_PATH)

required_zone_cols = ["zona_id", "nombre_zona", "isla", "municipio", "lat", "lon"]
missing_zone_cols = [c for c in required_zone_cols if c not in beach_geography.columns]

if missing_zone_cols:
    raise ValueError(f"Faltan columnas en beach_geography: {missing_zone_cols}")

print("beach_geography shape:", beach_geography.shape)
print("zona_id únicos:", beach_geography["zona_id"].nunique())
print("nulos lat:", beach_geography["lat"].isna().sum())
print("nulos lon:", beach_geography["lon"].isna().sum())

display(beach_geography.head())

gdf_zones = gpd.GeoDataFrame(
    beach_geography.copy(),
    geometry=gpd.points_from_xy(beach_geography["lon"], beach_geography["lat"]),
    crs="EPSG:4326",
)

gdf_zones_m = gdf_zones.to_crs("EPSG:3857")

beach_geography shape: (561, 17)
zona_id únicos: 561
nulos lat: 0
nulos lon: 0


,zona_id,nombre_zona,isla,municipio,lat,lon,tipo_zona,orientacion_costa,exposicion_norte,exposicion_oeste,exposicion_este,exposicion_swell_nw,exposicion_swell_ne,vulnerabilidad_costera,vulnerabilidad_source,spatial_match_isla,spatial_match_municipio
0,CAN_TF_EL_PUERTITO_0,El Puertito,Tenerife,Güímar,28.2923,-16.3766,playa,NW,1,1,0,1,1,no_disponible,/content/drive/MyDrive/AI Projects/DeepWave Ca...,True,True
1,CAN_EH_LA_RESTINGA,La Restinga,El Hierro,El Pinar de El Hierro,27.6408,-17.9799,playa,W,0,1,0,1,0,no_disponible,/content/drive/MyDrive/AI Projects/DeepWave Ca...,True,True
2,CAN_EH_ARENAS_BLANCAS,Arenas Blancas,El Hierro,Frontera,27.7667,-18.1218,playa,W,0,1,0,1,0,no_disponible,/content/drive/MyDrive/AI Projects/DeepWave Ca...,True,True
3,CAN_EH_EL_VERODAL,El Verodal,El Hierro,Frontera,27.7471,-18.1512,playa,W,0,1,0,1,0,no_disponible,/content/drive/MyDrive/AI Projects/DeepWave Ca...,True,True
4,CAN_EH_CHARCO_AZUL_0,Charco Azul,El Hierro,Frontera,27.7563,-18.0990,playa,W,0,1,0,1,0,no_disponible,/content/drive/MyDrive/AI Projects/DeepWave Ca...,True,True


## Celda 5 — Localizar archivos SIMAR y parsear nombres

In [45]:
simar_files = sorted(SIMAR_DIR.glob("*.csv"))

if not simar_files:
    raise FileNotFoundError(f"No se encontraron CSV en {SIMAR_DIR}")

FILENAME_RE = re.compile(
    r"^(?P<request_id>\d+)_(?P<download_id>\d+)_(?P<point_id>\d+)_(?P<variable_group>[A-Z]+(?:_[A-Z]+)*)_(?P<start>\d{14})_(?P<end>\d{14})\.csv$",
    re.IGNORECASE,
)


def parse_simar_filename(path):
    m = FILENAME_RE.match(path.name)

    if not m:
        return {
            "filename": path.name,
            "request_id": None,
            "download_id": None,
            "simar_point_id": None,
            "variable_group": "UNKNOWN",
            "file_start": pd.NaT,
            "file_end": pd.NaT,
            "filename_parse_ok": False,
        }

    d = m.groupdict()

    return {
        "filename": path.name,
        "request_id": d["request_id"],
        "download_id": d["download_id"],
        "simar_point_id": d["point_id"],
        "variable_group": d["variable_group"].upper(),
        "file_start": pd.to_datetime(d["start"], format="%Y%m%d%H%M%S", errors="coerce", utc=True),
        "file_end": pd.to_datetime(d["end"], format="%Y%m%d%H%M%S", errors="coerce", utc=True),
        "filename_parse_ok": True,
    }


files_df = pd.DataFrame([parse_simar_filename(p) | {"path": str(p)} for p in simar_files])

print("Archivos SIMAR encontrados:", len(files_df))
display(files_df)

print("Conteo por variable_group:")
display(files_df["variable_group"].value_counts().reset_index())

Archivos SIMAR encontrados: 71


,filename,request_id,download_id,simar_point_id,variable_group,file_start,file_end,filename_parse_ok,path
0,25408_52281_4030024_WAVE_20250506172439_202605...,25408,52281,4030024,WAVE,2025-05-06 17:24:39+00:00,2026-05-06 17:24:39+00:00,True,/content/drive/MyDrive/AI Projects/DeepWave Ca...
1,25408_52282_4030024_CURRENTS_20250506172439_20...,25408,52282,4030024,CURRENTS,2025-05-06 17:24:39+00:00,2026-05-06 17:24:39+00:00,True,/content/drive/MyDrive/AI Projects/DeepWave Ca...
2,25408_52283_4030024_WATER_TEMP_20250506172439_...,25408,52283,4030024,WATER_TEMP,2025-05-06 17:24:39+00:00,2026-05-06 17:24:39+00:00,True,/content/drive/MyDrive/AI Projects/DeepWave Ca...
3,25408_52284_4030024_SALINITY_20250506172439_20...,25408,52284,4030024,SALINITY,2025-05-06 17:24:39+00:00,2026-05-06 17:24:39+00:00,True,/content/drive/MyDrive/AI Projects/DeepWave Ca...
4,25408_52285_4030024_WIND_20250506172439_202605...,25408,52285,4030024,WIND,2025-05-06 17:24:39+00:00,2026-05-06 17:24:39+00:00,True,/content/drive/MyDrive/AI Projects/DeepWave Ca...
...,...,...,...,...,...,...,...,...,...
66,25410_52351_4058026_WAVE_20010101183202_202605...,25410,52351,4058026,WAVE,2001-01-01 18:32:02+00:00,2026-05-06 17:32:02+00:00,True,/content/drive/MyDrive/AI Projects/DeepWave Ca...
67,25410_52352_4058026_CURRENTS_20010101183202_20...,25410,52352,4058026,CURRENTS,2001-01-01 18:32:02+00:00,2026-05-06 17:32:02+00:00,True,/content/drive/MyDrive/AI Projects/DeepWave Ca...
68,25410_52353_4058026_WATER_TEMP_20010101183202_...,25410,52353,4058026,WATER_TEMP,2001-01-01 18:32:02+00:00,2026-05-06 17:32:02+00:00,True,/content/drive/MyDrive/AI Projects/DeepWave Ca...
69,25410_52354_4058026_SALINITY_20010101183202_20...,25410,52354,4058026,SALINITY,2001-01-01 18:32:02+00:00,2026-05-06 17:32:02+00:00,True,/content/drive/MyDrive/AI Projects/DeepWave Ca...


Conteo por variable_group:


,variable_group,count
0,WATER_TEMP,15
1,SALINITY,15
2,WAVE,14
3,CURRENTS,14
4,WIND,13


## Celda 6 — Lectura robusta de CSV propietario de Puertos del Estado

In [46]:
def read_text_lines(path, encodings=("utf-8", "utf-8-sig", "latin1", "cp1252")):
    last_error = None

    for enc in encodings:
        try:
            with open(path, "r", encoding=enc, errors="replace") as f:
                return f.readlines(), enc
        except Exception as e:
            last_error = e

    raise last_error


def detect_delimiter(lines, sample_size=80):
    candidates = [";", ",", "\t", "|"]
    scores = {sep: 0 for sep in candidates}

    for line in lines[:sample_size]:
        for sep in candidates:
            scores[sep] += line.count(sep)

    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else ";"


def detect_table_start(lines, delimiter):
    """Intenta localizar la fila de cabecera tabular."""
    date_keywords = ["FECHA", "DATE", "HORA", "TIME", "GMT", "UTC"]

    for i, line in enumerate(lines[:400]):
        norm = normalize_text(line)
        if pd.isna(norm):
            continue

        has_keyword = any(k in norm for k in date_keywords)
        has_delim = line.count(delimiter) >= 1

        if has_keyword and has_delim:
            return i

    # Fallback: primera línea que parece dato temporal.
    for i, line in enumerate(lines[:400]):
        if line.count(delimiter) >= 1 and looks_like_date_string(line):
            if i > 0 and lines[i - 1].count(delimiter) >= 1 and not looks_like_date_string(lines[i - 1]):
                return i - 1
            return i

    # Último fallback: primera línea con varios delimitadores.
    for i, line in enumerate(lines[:400]):
        if line.count(delimiter) >= 2:
            return i

    return 0


def extract_coords_from_metadata(metadata_lines):
    """Extrae lat/lon desde líneas de metadatos."""
    lat = np.nan
    lon = np.nan

    for line in metadata_lines:
        norm = normalize_text(line)
        if pd.isna(norm):
            continue

        if "LAT" in norm and pd.isna(lat):
            candidate = parse_coordinate(line)
            if 20 <= candidate <= 40:
                lat = candidate

        if ("LON" in norm or "LONG" in norm) and pd.isna(lon):
            candidate = parse_coordinate(line)
            if -30 <= candidate <= 0 or 0 <= candidate <= 30:
                lon = candidate
                if lon > 0 and any(h in norm for h in [" W", " O", "OESTE", "WEST"]):
                    lon = -lon

    return lat, lon


def extract_coords_from_dataframe(df):
    """Busca lat/lon en columnas de datos si no aparecieron en metadatos."""
    lat_col = infer_column(
        df,
        [
            "lat",
            "latitude",
            "latitud",
            "coord_lat",
            "coordenada_latitud",
        ],
    )

    lon_col = infer_column(
        df,
        [
            "lon",
            "lng",
            "longitude",
            "longitud",
            "coord_lon",
            "coordenada_longitud",
        ],
    )

    lat = np.nan
    lon = np.nan

    if lat_col is not None:
        vals = df[lat_col].apply(parse_coordinate)
        vals = vals[vals.between(20, 40)]
        if len(vals):
            lat = float(vals.iloc[0])

    if lon_col is not None:
        vals = df[lon_col].apply(parse_coordinate)
        vals = vals[vals.between(-30, 0)]
        if len(vals):
            lon = float(vals.iloc[0])

    return lat, lon


def read_puertos_csv(path):
    """Lee un CSV de Puertos del Estado con cabecera propietaria."""
    lines, encoding = read_text_lines(path)
    delimiter = detect_delimiter(lines)
    table_start = detect_table_start(lines, delimiter)
    metadata_lines = lines[:table_start]

    read_kwargs = dict(
        sep=delimiter,
        skiprows=table_start,
        encoding=encoding,
        engine="python",
        on_bad_lines="skip",
    )

    df = pd.read_csv(path, **read_kwargs)

    # Si la primera fila de datos fue tomada como cabecera, rehacemos lectura sin header.
    if any(looks_like_date_string(c) for c in df.columns):
        df = pd.read_csv(path, header=None, **read_kwargs)
        df.columns = [f"col_{i}" for i in range(df.shape[1])]

    # Limpieza básica de columnas.
    df = df.dropna(axis=1, how="all")
    df.columns = [str(c).strip() for c in df.columns]

    lat_meta, lon_meta = extract_coords_from_metadata(metadata_lines)
    lat_df, lon_df = extract_coords_from_dataframe(df)

    lat = lat_meta if not pd.isna(lat_meta) else lat_df
    lon = lon_meta if not pd.isna(lon_meta) else lon_df

    meta = {
        "encoding": encoding,
        "delimiter": delimiter,
        "table_start_line": table_start,
        "metadata_line_count": len(metadata_lines),
        "lat": lat,
        "lon": lon,
        "raw_columns": list(df.columns),
    }

    return df, meta

## Celda 7 — Estandarización de columnas SIMAR

In [47]:
TIMESTAMP_EXCLUDE = [r"LAT", r"LON", r"LONG", r"POINT", r"PUNTO", r"ID", r"ESTACION"]

# Rangos temporales válidos para este proyecto.
MIN_VALID_TS = pd.Timestamp("1990-01-01", tz="UTC")
MAX_VALID_TS = pd.Timestamp("2030-12-31", tz="UTC")


def parse_datetime_series(series):
    """
    Parsea una serie temporal evitando el error típico de pandas:
    convertir enteros 0,1,2... en nanosegundos desde 1970.

    Solo acepta fechas con año real entre MIN_VALID_TS y MAX_VALID_TS.
    """
    raw = series.astype(str).str.strip()
    out = pd.Series(pd.NaT, index=series.index, dtype="datetime64[ns, UTC]")

    empty_mask = raw.str.upper().isin(["", "NAN", "NONE", "NULL", "NA", "N/A"])
    raw = raw.mask(empty_mask)

    # 1) Formatos compactos tipo YYYYMMDD, YYYYMMDDHH, YYYYMMDDHHMM, YYYYMMDDHHMMSS.
    compact = raw.str.replace(r"\.0$", "", regex=True)
    compact_mask = compact.str.fullmatch(r"\d{8}|\d{10}|\d{12}|\d{14}", na=False)

    compact_formats = {
        8: "%Y%m%d",
        10: "%Y%m%d%H",
        12: "%Y%m%d%H%M",
        14: "%Y%m%d%H%M%S",
    }

    for length, fmt in compact_formats.items():
        mask = compact_mask & compact.str.len().eq(length)
        if mask.any():
            parsed = pd.to_datetime(compact[mask], format=fmt, errors="coerce", utc=True)
            out.loc[mask] = parsed

    # 2) Fechas con separadores.
    # Exigimos algún separador o letra T para no parsear columnas numéricas normales.
    sep_mask = out.isna() & raw.str.contains(r"[-/:T ]", regex=True, na=False)
    if sep_mask.any():
        parsed = pd.to_datetime(raw[sep_mask], errors="coerce", dayfirst=True, utc=True)
        out.loc[sep_mask] = parsed

    # 3) Filtrar años imposibles.
    valid_mask = out.between(MIN_VALID_TS, MAX_VALID_TS)
    out = out.where(valid_mask)

    return out


def is_good_timestamp(parsed, min_valid_ratio=0.5):
    if len(parsed) == 0:
        return False

    valid_ratio = parsed.notna().mean()

    if valid_ratio < min_valid_ratio:
        return False

    years = parsed.dropna().dt.year

    if years.empty:
        return False

    return years.between(MIN_VALID_TS.year, MAX_VALID_TS.year).all()


def detect_component_datetime(df):
    """
    Detecta fechas repartidas en columnas de año/mes/día/hora.
    """
    col_year = infer_column(df, ["año", "ano", "year", "yyyy"])
    col_month = infer_column(df, ["mes", "month", "mm"])
    col_day = infer_column(df, ["dia", "día", "day", "dd"])
    col_hour = infer_column(df, ["hora", "hour", "hh"])

    if col_year is None or col_month is None or col_day is None:
        return None, []

    y = to_numeric_series(df[col_year])
    m = to_numeric_series(df[col_month])
    d = to_numeric_series(df[col_day])

    if col_hour is not None:
        h = to_numeric_series(df[col_hour]).fillna(0)
    else:
        h = pd.Series(0, index=df.index)

    candidate = pd.DataFrame(
        {
            "year": y,
            "month": m,
            "day": d,
            "hour": h,
        }
    )

    parsed = pd.to_datetime(candidate, errors="coerce", utc=True)
    parsed = parsed.where(parsed.between(MIN_VALID_TS, MAX_VALID_TS))

    used_cols = [c for c in [col_year, col_month, col_day, col_hour] if c is not None]

    if is_good_timestamp(parsed):
        return parsed, used_cols

    return None, []


def detect_timestamp(df):
    """
    Devuelve una Serie timestamp UTC.
    Evita aceptar timestamps de 1970 generados por columnas numéricas.
    """
    # 0) Año/mes/día/hora separados.
    parsed_components, used_component_cols = detect_component_datetime(df)
    if parsed_components is not None:
        return parsed_components, used_component_cols

    date_cols = []
    time_cols = []

    for col in df.columns:
        norm = normalize_col(col)

        if any(re.search(ex, norm) for ex in TIMESTAMP_EXCLUDE):
            continue

        if (
            "FECHA" in norm
            or "DATE" in norm
            or norm in ["DIA", "DAY", "TIMESTAMP", "DATETIME"]
        ):
            date_cols.append(col)

        if (
            "HORA" in norm
            or "TIME" in norm
            or norm in ["HH", "HOUR"]
        ):
            time_cols.append(col)

    # 1) Fecha + hora separadas.
    for dcol in date_cols:
        if time_cols:
            for hcol in time_cols:
                candidate = df[dcol].astype(str).str.strip() + " " + df[hcol].astype(str).str.strip()
                parsed = parse_datetime_series(candidate)
                if is_good_timestamp(parsed):
                    return parsed, [dcol, hcol]

        parsed = parse_datetime_series(df[dcol])
        if is_good_timestamp(parsed):
            return parsed, [dcol]

    # 2) Buscar columnas con contenido claramente temporal.
    for col in df.columns:
        norm = normalize_col(col)

        if any(re.search(ex, norm) for ex in TIMESTAMP_EXCLUDE):
            continue

        s = df[col].astype(str).str.strip()
        has_date_like_content = (
            s.str.contains(r"\d{1,4}[-/]\d{1,2}[-/]\d{1,4}", regex=True, na=False).mean() > 0.3
            or s.str.replace(r"\.0$", "", regex=True).str.fullmatch(r"\d{8}|\d{10}|\d{12}|\d{14}", na=False).mean() > 0.3
        )

        if not has_date_like_content:
            continue

        parsed = parse_datetime_series(df[col])
        if is_good_timestamp(parsed):
            return parsed, [col]

    # 3) Fallback: primeras dos columnas combinadas, solo si parecen fecha/hora.
    if df.shape[1] >= 2:
        c0 = df.iloc[:, 0].astype(str).str.strip()
        c1 = df.iloc[:, 1].astype(str).str.strip()
        candidate = c0 + " " + c1

        parsed = parse_datetime_series(candidate)
        if is_good_timestamp(parsed):
            return parsed, [df.columns[0], df.columns[1]]

    raise ValueError("No se pudo detectar columna temporal válida. No se aceptan timestamps 1970.")


VARIABLE_PATTERNS = {
    "WAVE": {
        "hs": [
            r"ALTURA.*SIGNIF.*OLEAJE",
            r"ALT.*SIGN.*OLEAJE",
            r"(^|_)HS($|_)",
            r"(^|_)H_S($|_)",
            r"HM0",
            r"HMO",
            r"VHM0",
        ],
        "hmax": [
            r"HMAX",
            r"H_MAX",
            r"MAX.*OLA",
            r"ALT.*MAX",
        ],
        "tp": [
            r"PERIODO.*PICO(?!.*FONDO)",
            r"PER.*PICO(?!.*FONDO)",
            r"(^|_)TP($|_)",
            r"TPK",
            r"VTPK",
        ],
        "tm02": [
            r"TM02(?!.*VIENTO)(?!.*FONDO)",
            r"T_M02",
            r"PERIODO.*MEDIO.*TM02(?!.*VIENTO)(?!.*FONDO)",
            r"PER.*MED",
            r"MEAN.*PER",
            r"TZ",
        ],
        "wave_direction": [
            r"DIR.*MEDIA.*OLEAJE",
            r"DIREC.*OLEAJE",
            r"DIRECTION.*WAVE",
            r"VMDR",
            r"MWD",
        ],
        "swell_height": [
            r"MAR.*FONDO.*ALTURA",
            r"MAR.*FONDO.*ALT",
            r"SWELL.*H",
            r"VHM0_SW",
        ],
        "swell_period": [
            r"MAR.*FONDO.*PERIODO",
            r"MAR.*FONDO.*PER",
            r"SWELL.*PER",
            r"VTPK_SW",
        ],
        "swell_direction": [
            r"MAR.*FONDO.*DIR",
            r"MAR.*FONDO.*DIREC",
            r"SWELL.*DIR",
            r"VMDR_SW",
        ],
        "wind_wave_height": [
            r"MAR.*VIENTO.*ALTURA",
            r"MAR.*VIENTO.*ALT",
            r"WIND.*WAVE.*H",
            r"SEA.*H",
            r"VHM0_WW",
        ],
        "wind_wave_period": [
            r"MAR.*VIENTO.*PERIODO",
            r"MAR.*VIENTO.*PER",
            r"WIND.*WAVE.*PER",
            r"SEA.*PER",
            r"VTPK_WW",
        ],
    },
    "WIND": {
        "wind_speed": [
            r"VELOCIDAD.*VIENTO",
            r"VEL.*VIENTO",
            r"WIND.*SPEED",
            r"(^|_)VMED($|_)",
            r"(^|_)VEL($|_)",
            r"SPEED",
        ],
        "wind_direction": [
            r"DIRECCION.*VIENTO",
            r"DIR.*VIENTO",
            r"WIND.*DIR",
            r"DIREC",
            r"DIRECTION",
            r"(^|_)DIR($|_)",
        ],
        "wind_gust": [
            r"GUST",
            r"RACHA",
            r"VMAX",
            r"V_MAX",
            r"MAX.*VIENTO",
        ],
    },
    "CURRENTS": {
        "current_u": [
            r"(^|_)U($|_)",
            r"UCUR",
            r"COMP.*U",
            r"EAST",
            r"ZONAL",
        ],
        "current_v": [
            r"(^|_)V($|_)",
            r"VCUR",
            r"COMP.*V",
            r"NORTH",
            r"MERID",
        ],
        "current_speed": [
            r"VELOCIDAD.*CORRIENTE",
            r"VEL.*CORR",
            r"CORRIENTE.*VELOCIDAD",
            r"CURRENT.*SPEED",
            r"SPEED",
            r"MAG",
        ],
        "current_direction": [
            r"DIRECCION.*CORRIENTE",
            r"DIR.*CORR",
            r"CORRIENTE.*DIREC",
            r"CURRENT.*DIR",
            r"DIRECTION",
        ],
    },
    "WATER_TEMP": {
        "sea_surface_temperature": [
            r"TEMPERATURA.*AGUA",
            r"WATER.*TEMP",
            r"TEMPERATURA",
            r"(^|_)TEMP($|_)",
            r"SST",
            r"THETA",
        ],
    },
    "SALINITY": {
        "sea_surface_salinity": [
            r"SALINIDAD",
            r"SALINITY",
            r"(^|_)SAL($|_)",
            r"(^|_)SO($|_)",
        ],
    },
}


VARIABLE_EXCLUDES = {
    "hs": [r"VIENTO", r"FONDO", r"PERIODO", r"DIR", r"DIREC"],
    "hmax": [r"PERIODO", r"DIR", r"DIREC"],
    "tp": [r"DIR", r"DIREC"],
    "tm02": [r"DIR", r"DIREC"],
    "wave_direction": [r"PERIODO", r"ALTURA", r"VIENTO", r"FONDO"],
    "swell_height": [r"PERIODO", r"DIR", r"DIREC"],
    "swell_period": [r"ALTURA", r"DIR", r"DIREC"],
    "swell_direction": [r"ALTURA", r"PERIODO"],
    "wind_wave_height": [r"PERIODO", r"DIR", r"DIREC"],
    "wind_wave_period": [r"ALTURA", r"DIR", r"DIREC"],
    "wind_speed": [r"DIR", r"DIREC"],
    "wind_direction": [r"VELOCIDAD", r"SPEED", r"RACHA", r"GUST"],
    "wind_gust": [r"DIR", r"DIREC"],
    "current_speed": [r"DIR", r"DIREC"],
    "current_direction": [r"VELOCIDAD", r"SPEED"],
}


def map_variable_columns(df, variable_group, timestamp_cols):
    """
    Mapea columnas raw a nombres canónicos según el grupo de variable.
    """
    variable_group = variable_group.upper()
    patterns = VARIABLE_PATTERNS.get(variable_group, {})

    mapped = {}
    used_cols = set(timestamp_cols)

    common_excludes = [
        r"FECHA",
        r"DATE",
        r"HORA",
        r"TIME",
        r"LAT",
        r"LON",
        r"LONG",
        r"PUNTO",
        r"POINT",
        r"ID",
    ]

    for canonical_col, pats in patterns.items():
        excludes = common_excludes + VARIABLE_EXCLUDES.get(canonical_col, [])

        col = find_col_by_patterns(
            df,
            pats,
            exclude_patterns=excludes,
        )

        if col is not None and col not in used_cols:
            mapped[canonical_col] = col
            used_cols.add(col)

    # Fallback para archivos de una sola variable con columna genérica.
    numeric_candidate_cols = []
    for col in df.columns:
        if col in used_cols:
            continue
        norm = normalize_col(col)
        if any(re.search(ex, norm) for ex in common_excludes):
            continue
        values = to_numeric_series(df[col])
        if values.notna().mean() > 0.5:
            numeric_candidate_cols.append(col)

    if variable_group == "WATER_TEMP" and "sea_surface_temperature" not in mapped and len(numeric_candidate_cols) == 1:
        mapped["sea_surface_temperature"] = numeric_candidate_cols[0]

    if variable_group == "SALINITY" and "sea_surface_salinity" not in mapped and len(numeric_candidate_cols) == 1:
        mapped["sea_surface_salinity"] = numeric_candidate_cols[0]

    return mapped, numeric_candidate_cols


def standardize_simar_dataframe(raw_df, file_meta, filename_meta):
    """
    Convierte un CSV raw SIMAR a un DataFrame con columnas canónicas.
    """
    df = raw_df.copy()
    variable_group = filename_meta["variable_group"]
    simar_point_id = filename_meta["simar_point_id"]

    timestamp, timestamp_cols = detect_timestamp(df)

    if not is_good_timestamp(timestamp, min_valid_ratio=0.8):
        raise ValueError(
            f"Timestamps inválidos en {filename_meta['filename']}. "
            f"Rango detectado: {timestamp.min()} - {timestamp.max()}"
        )

    mapped_cols, numeric_candidates = map_variable_columns(
        df,
        variable_group=variable_group,
        timestamp_cols=timestamp_cols,
    )

    out = pd.DataFrame()
    out["timestamp"] = timestamp
    out["simar_point_id"] = str(simar_point_id)
    out["variable_group"] = variable_group
    out["source_file"] = filename_meta["filename"]

    for canonical_col, raw_col in mapped_cols.items():
        out[canonical_col] = to_numeric_series(df[raw_col])

    # Coordenadas del punto.
    lat = file_meta.get("lat", np.nan)
    lon = file_meta.get("lon", np.nan)

    if pd.isna(lat) or pd.isna(lon):
        lat_df, lon_df = extract_coords_from_dataframe(df)
        if pd.isna(lat):
            lat = lat_df
        if pd.isna(lon):
            lon = lon_df

    out["point_lat"] = lat
    out["point_lon"] = lon

    # Limpieza de timestamps inválidos.
    before = len(out)
    out = out.dropna(subset=["timestamp"]).copy()
    dropped_bad_timestamp = before - len(out)

    # Normalización básica de direcciones.
    for dir_col in [
        "wave_direction",
        "swell_direction",
        "wind_direction",
        "current_direction",
    ]:
        if dir_col in out.columns:
            out[dir_col] = out[dir_col] % 360

    # Temperatura: si viniera en Kelvin, convertir a Celsius.
    if "sea_surface_temperature" in out.columns:
        med = out["sea_surface_temperature"].median(skipna=True)
        if pd.notna(med) and med > 100:
            out["sea_surface_temperature"] = out["sea_surface_temperature"] - 273.15

    # Corriente: calcular velocidad/dirección si hay u/v.
    if variable_group == "CURRENTS":
        if "current_speed" not in out.columns and {"current_u", "current_v"}.issubset(out.columns):
            out["current_speed"] = np.sqrt(out["current_u"] ** 2 + out["current_v"] ** 2)

        if "current_direction" not in out.columns and {"current_u", "current_v"}.issubset(out.columns):
            # Dirección hacia donde fluye la corriente, grados desde el norte.
            out["current_direction"] = (np.degrees(np.arctan2(out["current_u"], out["current_v"])) + 360) % 360

    summary = {
        "filename": filename_meta["filename"],
        "simar_point_id": simar_point_id,
        "variable_group": variable_group,
        "raw_rows": len(raw_df),
        "standardized_rows": len(out),
        "dropped_bad_timestamp": dropped_bad_timestamp,
        "timestamp_min": out["timestamp"].min() if len(out) else pd.NaT,
        "timestamp_max": out["timestamp"].max() if len(out) else pd.NaT,
        "point_lat": lat,
        "point_lon": lon,
        "encoding": file_meta.get("encoding"),
        "delimiter": file_meta.get("delimiter"),
        "table_start_line": file_meta.get("table_start_line"),
        "timestamp_columns": json.dumps(timestamp_cols, ensure_ascii=False),
        "mapped_columns": json.dumps(mapped_cols, ensure_ascii=False),
        "numeric_candidate_columns": json.dumps(numeric_candidates, ensure_ascii=False),
        "raw_columns": json.dumps(file_meta.get("raw_columns", []), ensure_ascii=False),
    }

    return out, summary

In [48]:
MIN_VALID_TS = pd.Timestamp("1999-01-01", tz="UTC")
MAX_VALID_TS = pd.Timestamp("2031-01-01", tz="UTC")


def is_good_timestamp(timestamp, min_valid_ratio=0.8):
    """
    Comprueba que una serie temporal tiene suficientes fechas válidas
    y que no cae en falsos años 1969/1970.
    """
    ts = pd.to_datetime(timestamp, utc=True, errors="coerce")

    if len(ts) == 0:
        return False

    valid_ratio = ts.notna().mean()

    if valid_ratio < min_valid_ratio:
        return False

    ts_valid = ts.dropna()

    if ts_valid.empty:
        return False

    inside_range_ratio = ts_valid.between(MIN_VALID_TS, MAX_VALID_TS).mean()

    if inside_range_ratio < min_valid_ratio:
        return False

    return True


def build_timestamp_from_filename(filename_meta, n_rows):
    """
    Fallback para SIMAR cuando el CSV no permite detectar bien la columna temporal.
    Usa file_start/file_end del nombre del archivo y genera una secuencia horaria.

    Ejemplo:
    25409_52306_4048030_WAVE_20000101182758_20260506172758.csv
    """

    start = filename_meta.get("file_start", pd.NaT)
    end = filename_meta.get("file_end", pd.NaT)

    if pd.isna(start) or pd.isna(end):
        raise ValueError(
            f"No se puede reconstruir timestamp desde nombre para {filename_meta['filename']}"
        )

    start = pd.to_datetime(start, utc=True).floor("h")
    end = pd.to_datetime(end, utc=True).floor("h")

    if start < MIN_VALID_TS or end > MAX_VALID_TS:
        raise ValueError(
            f"Fechas del filename fuera de rango en {filename_meta['filename']}: {start} - {end}"
        )

    if n_rows <= 0:
        return pd.Series([], dtype="datetime64[ns, UTC]"), ["filename_start_hourly_fallback"]

    candidate = pd.date_range(
        start=start,
        periods=n_rows,
        freq="h",
        tz="UTC",
    )

    diff_hours = abs((candidate[-1] - end).total_seconds()) / 3600

    if diff_hours > 48:
        print(
            f"AVISO timestamp fallback {filename_meta['filename']}: "
            f"último_generado={candidate[-1]}, end_filename={end}, diff_h={diff_hours:.1f}"
        )

    return pd.Series(candidate, dtype="datetime64[ns, UTC]"), ["filename_start_hourly_fallback"]


def standardize_simar_dataframe(raw_df, file_meta, filename_meta):
    """
    Convierte un CSV raw SIMAR a un DataFrame con columnas canónicas.
    Si no detecta timestamp válido en columnas, reconstruye timestamp horario
    desde el nombre del archivo.
    """

    df = raw_df.copy()
    variable_group = filename_meta["variable_group"]
    simar_point_id = filename_meta["simar_point_id"]

    timestamp_source = "csv_columns"

    try:
        timestamp, timestamp_cols = detect_timestamp(df)

        if not is_good_timestamp(timestamp, min_valid_ratio=0.8):
            raise ValueError(
                f"Timestamps detectados inválidos. "
                f"Rango={pd.to_datetime(timestamp, utc=True, errors='coerce').min()} - "
                f"{pd.to_datetime(timestamp, utc=True, errors='coerce').max()}"
            )

        timestamp = pd.to_datetime(timestamp, utc=True, errors="coerce").reset_index(drop=True)

    except Exception as e:
        print(
            f"AVISO: timestamp no detectable/válido en {filename_meta['filename']}. "
            f"Usando fallback desde filename. Error original: {repr(e)}"
        )

        timestamp, timestamp_cols = build_timestamp_from_filename(
            filename_meta=filename_meta,
            n_rows=len(df),
        )

        timestamp = pd.to_datetime(timestamp, utc=True, errors="coerce").reset_index(drop=True)
        timestamp_source = "filename_hourly_fallback"

    mapped_cols, numeric_candidates = map_variable_columns(
        df,
        variable_group=variable_group,
        timestamp_cols=timestamp_cols,
    )

    out = pd.DataFrame()
    out["timestamp"] = timestamp
    out["simar_point_id"] = str(simar_point_id)
    out["variable_group"] = variable_group
    out["source_file"] = filename_meta["filename"]

    for canonical_col, raw_col in mapped_cols.items():
        out[canonical_col] = to_numeric_series(df[raw_col]).reset_index(drop=True)

    lat = file_meta.get("lat", np.nan)
    lon = file_meta.get("lon", np.nan)

    if pd.isna(lat) or pd.isna(lon):
        lat_df, lon_df = extract_coords_from_dataframe(df)

        if pd.isna(lat):
            lat = lat_df

        if pd.isna(lon):
            lon = lon_df

    out["point_lat"] = lat
    out["point_lon"] = lon

    before = len(out)
    out = out.dropna(subset=["timestamp"]).copy()
    dropped_bad_timestamp = before - len(out)

    invalid_time = ~out["timestamp"].between(MIN_VALID_TS, MAX_VALID_TS)

    if invalid_time.any():
        raise ValueError(
            f"{filename_meta['filename']} contiene timestamps fuera de rango: "
            f"{out.loc[invalid_time, 'timestamp'].min()} - "
            f"{out.loc[invalid_time, 'timestamp'].max()}"
        )

    for dir_col in [
        "wave_direction",
        "swell_direction",
        "wind_direction",
        "current_direction",
    ]:
        if dir_col in out.columns:
            out[dir_col] = out[dir_col] % 360

    if "sea_surface_temperature" in out.columns:
        med = out["sea_surface_temperature"].median(skipna=True)
        if pd.notna(med) and med > 100:
            out["sea_surface_temperature"] = out["sea_surface_temperature"] - 273.15

    if variable_group == "CURRENTS":
        if "current_speed" not in out.columns and {"current_u", "current_v"}.issubset(out.columns):
            out["current_speed"] = np.sqrt(out["current_u"] ** 2 + out["current_v"] ** 2)

        if "current_direction" not in out.columns and {"current_u", "current_v"}.issubset(out.columns):
            out["current_direction"] = (
                np.degrees(np.arctan2(out["current_u"], out["current_v"])) + 360
            ) % 360

    summary = {
        "filename": filename_meta["filename"],
        "simar_point_id": simar_point_id,
        "variable_group": variable_group,
        "raw_rows": len(raw_df),
        "standardized_rows": len(out),
        "dropped_bad_timestamp": dropped_bad_timestamp,
        "timestamp_min": out["timestamp"].min() if len(out) else pd.NaT,
        "timestamp_max": out["timestamp"].max() if len(out) else pd.NaT,
        "timestamp_source": timestamp_source,
        "point_lat": lat,
        "point_lon": lon,
        "encoding": file_meta.get("encoding"),
        "delimiter": file_meta.get("delimiter"),
        "table_start_line": file_meta.get("table_start_line"),
        "timestamp_columns": json.dumps(timestamp_cols, ensure_ascii=False),
        "mapped_columns": json.dumps(mapped_cols, ensure_ascii=False),
        "numeric_candidate_columns": json.dumps(numeric_candidates, ensure_ascii=False),
        "raw_columns": json.dumps(file_meta.get("raw_columns", []), ensure_ascii=False),
    }

    return out, summary


print("Parche 7B cargado: standardize_simar_dataframe queda sobrescrita con fallback temporal.")

Parche 7B cargado: standardize_simar_dataframe queda sobrescrita con fallback temporal.


In [49]:
def parse_compact_datetime_series(series):
    """
    Detecta fechas compactas tipo:
    YYYYMMDDHH
    YYYYMMDDHHMM
    YYYYMMDDHHMMSS

    Evita que pandas interprete enteros como nanosegundos Unix 1970.
    """

    raw = series.astype(str).str.strip()
    raw = raw.str.replace(r"\.0$", "", regex=True)
    digits = raw.str.replace(r"\D", "", regex=True)

    candidates = [
        (14, "%Y%m%d%H%M%S"),
        (12, "%Y%m%d%H%M"),
        (10, "%Y%m%d%H"),
        (8, "%Y%m%d"),
    ]

    best_parsed = None
    best_ratio = 0

    for length, fmt in candidates:
        mask = digits.str.len() == length

        if mask.mean() < 0.5:
            continue

        parsed = pd.to_datetime(
            digits.where(mask),
            format=fmt,
            errors="coerce",
            utc=True,
        )

        ratio = parsed.notna().mean()

        if ratio > best_ratio:
            best_ratio = ratio
            best_parsed = parsed

    if best_parsed is not None and is_good_timestamp(best_parsed, min_valid_ratio=0.8):
        return best_parsed

    return None


def detect_timestamp(df):
    """
    Detecta timestamp UTC.
    Versión corregida:
    - prioriza fechas compactas YYYYMMDDHH
    - evita falsos años 1969/1970
    - soporta fecha/hora separadas
    """

    date_cols = []
    time_cols = []

    for col in df.columns:
        norm = normalize_col(col)

        if any(re.search(ex, norm) for ex in TIMESTAMP_EXCLUDE):
            continue

        if "FECHA" in norm or "DATE" in norm or norm in ["DIA", "DAY"]:
            date_cols.append(col)

        if "HORA" in norm or "TIME" in norm or norm in ["HH", "H"]:
            time_cols.append(col)

    # Primero: columnas de fecha explícitas.
    for dcol in date_cols:
        compact = parse_compact_datetime_series(df[dcol])

        if compact is not None:
            return compact, [dcol]

        if time_cols:
            for hcol in time_cols:
                candidate = df[dcol].astype(str).str.strip() + " " + df[hcol].astype(str).str.strip()

                compact = parse_compact_datetime_series(candidate)

                if compact is not None:
                    return compact, [dcol, hcol]

                parsed = pd.to_datetime(
                    candidate,
                    errors="coerce",
                    dayfirst=True,
                    utc=True,
                )

                if is_good_timestamp(parsed, min_valid_ratio=0.8):
                    return parsed, [dcol, hcol]

        parsed = pd.to_datetime(
            df[dcol],
            errors="coerce",
            dayfirst=True,
            utc=True,
        )

        if is_good_timestamp(parsed, min_valid_ratio=0.8):
            return parsed, [dcol]

    # Segundo: cualquier columna que parezca fecha compacta.
    for col in df.columns:
        norm = normalize_col(col)

        if any(re.search(ex, norm) for ex in TIMESTAMP_EXCLUDE):
            continue

        compact = parse_compact_datetime_series(df[col])

        if compact is not None:
            return compact, [col]

    # Tercero: parse genérico, pero validado.
    for col in df.columns:
        norm = normalize_col(col)

        if any(re.search(ex, norm) for ex in TIMESTAMP_EXCLUDE):
            continue

        parsed = pd.to_datetime(
            df[col],
            errors="coerce",
            dayfirst=True,
            utc=True,
        )

        if is_good_timestamp(parsed, min_valid_ratio=0.8):
            return parsed, [col]

    # Cuarto: primeras dos columnas combinadas.
    if df.shape[1] >= 2:
        candidate = df.iloc[:, 0].astype(str).str.strip() + " " + df.iloc[:, 1].astype(str).str.strip()

        compact = parse_compact_datetime_series(candidate)

        if compact is not None:
            return compact, [df.columns[0], df.columns[1]]

        parsed = pd.to_datetime(
            candidate,
            errors="coerce",
            dayfirst=True,
            utc=True,
        )

        if is_good_timestamp(parsed, min_valid_ratio=0.8):
            return parsed, [df.columns[0], df.columns[1]]

    raise ValueError("No se pudo detectar columna temporal válida. No se aceptan timestamps 1970.")


# Ampliamos patrones para columnas españolas reales de SIMAR.
VARIABLE_PATTERNS["WAVE"]["wave_direction"] += [
    r"DIRECC",
    r"PROCED",
]

VARIABLE_PATTERNS["WAVE"]["swell_height"] += [
    r"MAR.*FONDO.*ALT",
]

VARIABLE_PATTERNS["WAVE"]["swell_period"] += [
    r"MAR.*FONDO.*PER",
]

VARIABLE_PATTERNS["WAVE"]["swell_direction"] += [
    r"MAR.*FONDO.*DIRECC",
    r"MAR.*FONDO.*PROCED",
]

VARIABLE_PATTERNS["WAVE"]["wind_wave_height"] += [
    r"MAR.*VIENTO.*ALT",
    r"VIENTO.*ALT",
]

VARIABLE_PATTERNS["WIND"]["wind_speed"] += [
    r"VELOCIDAD.*VIENTO",
    r"VEL.*VIENTO",
]

VARIABLE_PATTERNS["WIND"]["wind_direction"] += [
    r"DIREC.*VIENTO",
    r"PROCED.*VIENTO",
]

VARIABLE_PATTERNS["CURRENTS"]["current_speed"] += [
    r"VELOCIDAD.*CORRIENTE",
    r"VEL.*CORRIENTE",
]

VARIABLE_PATTERNS["CURRENTS"]["current_direction"] += [
    r"DIR.*CORRIENTE",
    r"PROP.*CORRIENTE",
]

VARIABLE_PATTERNS["SALINITY"]["sea_surface_salinity"] += [
    r"SALINIDAD",
    r"SALINIDAD.*PSU",
]


def map_variable_columns(df, variable_group, timestamp_cols):
    """
    Mapea columnas raw a nombres canónicos.
    Versión corregida: no excluye VELOCIDAD ni SALINIDAD por contener 'ID'.
    """

    variable_group = variable_group.upper()
    patterns = VARIABLE_PATTERNS.get(variable_group, {})

    mapped = {}
    used_cols = set(timestamp_cols)

    common_excludes = [
        r"^FECHA$",
        r"^FECHA_.*",
        r"^DATE$",
        r"^DATE_.*",
        r"^HORA$",
        r"^TIME$",
        r"^LAT$",
        r"^LATITUD$",
        r"^LON$",
        r"^LONGITUD$",
        r"^PUNTO$",
        r"^POINT$",
        r"^ID$",
        r"^ESTACION$",
    ]

    for canonical_col, pats in patterns.items():
        col = find_col_by_patterns(
            df,
            pats,
            exclude_patterns=common_excludes,
        )

        if col is not None and col not in used_cols:
            mapped[canonical_col] = col
            used_cols.add(col)

    numeric_candidate_cols = []

    for col in df.columns:
        if col in used_cols:
            continue

        norm = normalize_col(col)

        if any(re.search(ex, norm) for ex in common_excludes):
            continue

        values = to_numeric_series(df[col])

        if values.notna().mean() > 0.5:
            numeric_candidate_cols.append(col)

    if variable_group == "WATER_TEMP" and "sea_surface_temperature" not in mapped and len(numeric_candidate_cols) == 1:
        mapped["sea_surface_temperature"] = numeric_candidate_cols[0]

    if variable_group == "SALINITY" and "sea_surface_salinity" not in mapped and len(numeric_candidate_cols) == 1:
        mapped["sea_surface_salinity"] = numeric_candidate_cols[0]

    return mapped, numeric_candidate_cols


def standardize_simar_dataframe(raw_df, file_meta, filename_meta):
    """
    Estandarización SIMAR final:
    - timestamp real si existe
    - fallback desde filename si no se puede detectar
    - mapeo corregido de velocidad/salinidad
    - conversión de corriente cm/s a m/s
    """

    df = raw_df.copy()
    variable_group = filename_meta["variable_group"]
    simar_point_id = filename_meta["simar_point_id"]

    timestamp_source = "csv_columns"

    try:
        timestamp, timestamp_cols = detect_timestamp(df)

        if not is_good_timestamp(timestamp, min_valid_ratio=0.8):
            raise ValueError(
                f"Timestamps detectados inválidos. "
                f"Rango={pd.to_datetime(timestamp, utc=True, errors='coerce').min()} - "
                f"{pd.to_datetime(timestamp, utc=True, errors='coerce').max()}"
            )

        timestamp = pd.to_datetime(timestamp, utc=True, errors="coerce").reset_index(drop=True)

    except Exception as e:
        print(
            f"AVISO: timestamp no detectable/válido en {filename_meta['filename']}. "
            f"Usando fallback desde filename. Error original: {repr(e)}"
        )

        timestamp, timestamp_cols = build_timestamp_from_filename(
            filename_meta=filename_meta,
            n_rows=len(df),
        )

        timestamp = pd.to_datetime(timestamp, utc=True, errors="coerce").reset_index(drop=True)
        timestamp_source = "filename_hourly_fallback"

    mapped_cols, numeric_candidates = map_variable_columns(
        df,
        variable_group=variable_group,
        timestamp_cols=timestamp_cols,
    )

    out = pd.DataFrame()
    out["timestamp"] = timestamp
    out["simar_point_id"] = str(simar_point_id)
    out["variable_group"] = variable_group
    out["source_file"] = filename_meta["filename"]

    for canonical_col, raw_col in mapped_cols.items():
        values = to_numeric_series(df[raw_col]).reset_index(drop=True)

        raw_norm = normalize_col(raw_col)

        # Corrientes SIMAR suelen venir en cm/s. Silver debe estar en m/s.
        if canonical_col in ["current_speed", "current_u", "current_v"]:
            if "CM_S" in raw_norm or "CM" in raw_norm:
                values = values / 100.0

        out[canonical_col] = values

    lat = file_meta.get("lat", np.nan)
    lon = file_meta.get("lon", np.nan)

    if pd.isna(lat) or pd.isna(lon):
        lat_df, lon_df = extract_coords_from_dataframe(df)

        if pd.isna(lat):
            lat = lat_df

        if pd.isna(lon):
            lon = lon_df

    out["point_lat"] = lat
    out["point_lon"] = lon

    before = len(out)
    out = out.dropna(subset=["timestamp"]).copy()
    dropped_bad_timestamp = before - len(out)

    invalid_time = ~out["timestamp"].between(MIN_VALID_TS, MAX_VALID_TS)

    if invalid_time.any():
        raise ValueError(
            f"{filename_meta['filename']} contiene timestamps fuera de rango: "
            f"{out.loc[invalid_time, 'timestamp'].min()} - "
            f"{out.loc[invalid_time, 'timestamp'].max()}"
        )

    for dir_col in [
        "wave_direction",
        "swell_direction",
        "wind_direction",
        "current_direction",
    ]:
        if dir_col in out.columns:
            out[dir_col] = out[dir_col] % 360

    if "sea_surface_temperature" in out.columns:
        med = out["sea_surface_temperature"].median(skipna=True)

        if pd.notna(med) and med > 100:
            out["sea_surface_temperature"] = out["sea_surface_temperature"] - 273.15

    if variable_group == "CURRENTS":
        if "current_speed" not in out.columns and {"current_u", "current_v"}.issubset(out.columns):
            out["current_speed"] = np.sqrt(out["current_u"] ** 2 + out["current_v"] ** 2)

        if "current_direction" not in out.columns and {"current_u", "current_v"}.issubset(out.columns):
            out["current_direction"] = (
                np.degrees(np.arctan2(out["current_u"], out["current_v"])) + 360
            ) % 360

        # Si viene velocidad + dirección, derivamos u/v.
        if {"current_speed", "current_direction"}.issubset(out.columns):
            if "current_u" not in out.columns:
                current_rad = np.deg2rad(out["current_direction"])
                out["current_u"] = out["current_speed"] * np.sin(current_rad)

            if "current_v" not in out.columns:
                current_rad = np.deg2rad(out["current_direction"])
                out["current_v"] = out["current_speed"] * np.cos(current_rad)

    summary = {
        "filename": filename_meta["filename"],
        "simar_point_id": simar_point_id,
        "variable_group": variable_group,
        "raw_rows": len(raw_df),
        "standardized_rows": len(out),
        "dropped_bad_timestamp": dropped_bad_timestamp,
        "timestamp_min": out["timestamp"].min() if len(out) else pd.NaT,
        "timestamp_max": out["timestamp"].max() if len(out) else pd.NaT,
        "timestamp_source": timestamp_source,
        "point_lat": lat,
        "point_lon": lon,
        "encoding": file_meta.get("encoding"),
        "delimiter": file_meta.get("delimiter"),
        "table_start_line": file_meta.get("table_start_line"),
        "timestamp_columns": json.dumps(timestamp_cols, ensure_ascii=False),
        "mapped_columns": json.dumps(mapped_cols, ensure_ascii=False),
        "numeric_candidate_columns": json.dumps(numeric_candidates, ensure_ascii=False),
        "raw_columns": json.dumps(file_meta.get("raw_columns", []), ensure_ascii=False),
    }

    return out, summary


print("Parche 7C cargado: timestamp, velocidad, corriente y salinidad corregidos.")

Parche 7C cargado: timestamp, velocidad, corriente y salinidad corregidos.


## Celda 8 — Leer y estandarizar todos los archivos SIMAR

In [50]:
standardized_by_group = {}
file_summaries = []
read_errors = []

for _, row in tqdm(files_df.iterrows(), total=len(files_df), desc="Procesando SIMAR"):
    path = Path(row["path"])
    filename_meta = row.drop(labels=["path"]).to_dict()

    try:
        raw_df, file_meta = read_puertos_csv(path)
        std_df, summary = standardize_simar_dataframe(
            raw_df=raw_df,
            file_meta=file_meta,
            filename_meta=filename_meta,
        )

        group = filename_meta["variable_group"]
        standardized_by_group.setdefault(group, []).append(std_df)
        file_summaries.append(summary)

    except Exception as e:
        read_errors.append(
            {
                "filename": path.name,
                "path": str(path),
                "error": repr(e),
            }
        )

file_summary_df = pd.DataFrame(file_summaries)
read_errors_df = pd.DataFrame(read_errors)

print("Archivos procesados correctamente:", len(file_summary_df))
print("Errores de lectura:", len(read_errors_df))

if len(file_summary_df):
    display(file_summary_df)

if len(read_errors_df):
    display(read_errors_df)

# Guardar diagnóstico de lectura inmediatamente.
file_summary_df.to_csv(QC_DIR / "quality_simar_file_summary.csv", index=False)
read_errors_df.to_csv(QC_DIR / "quality_simar_read_errors.csv", index=False)

if len(read_errors_df) > 0:
    print("AVISO: hay archivos con error. Revisa quality_simar_read_errors.csv.")

Procesando SIMAR:   0%|          | 0/71 [00:00<?, ?it/s]

Archivos procesados correctamente: 71
Errores de lectura: 0


,filename,simar_point_id,variable_group,raw_rows,standardized_rows,dropped_bad_timestamp,timestamp_min,timestamp_max,timestamp_source,point_lat,point_lon,encoding,delimiter,table_start_line,timestamp_columns,mapped_columns,numeric_candidate_columns,raw_columns
0,25408_52281_4030024_WAVE_20250506172439_202605...,4030024,WAVE,8760,8760,0,2025-05-06 00:00:00+00:00,2026-05-06 23:00:00+00:00,csv_columns,NaN,NaN,utf-8,\t,1,"[""Fecha (GMT)""]","{""hs"": ""Altura Signif. del Oleaje(m)"", ""tp"": ""...","[""Mar de viento: Direcc. Media de Proced.(0=N,...","[""Fecha (GMT)"", ""Altura Signif. del Oleaje(m)""..."
1,25408_52282_4030024_CURRENTS_20250506172439_20...,4030024,CURRENTS,8712,8712,0,2025-05-06 00:00:00+00:00,2026-05-05 23:00:00+00:00,csv_columns,NaN,NaN,utf-8,\t,1,"[""Fecha (GMT)""]","{""current_speed"": ""Velocidad media de Corrient...",[],"[""Fecha (GMT)"", ""Velocidad media de Corriente(..."
2,25408_52283_4030024_WATER_TEMP_20250506172439_...,4030024,WATER_TEMP,8712,8712,0,2025-05-06 00:00:00+00:00,2026-05-05 23:00:00+00:00,csv_columns,NaN,NaN,utf-8,\t,1,"[""Fecha (GMT)""]","{""sea_surface_temperature"": ""Temperatura del A...",[],"[""Fecha (GMT)"", ""Temperatura del Agua(ºC)""]"
3,25408_52284_4030024_SALINITY_20250506172439_20...,4030024,SALINITY,8712,8712,0,2025-05-06 00:00:00+00:00,2026-05-05 23:00:00+00:00,csv_columns,NaN,NaN,utf-8,\t,1,"[""Fecha (GMT)""]","{""sea_surface_salinity"": ""Salinidad(psu)""}",[],"[""Fecha (GMT)"", ""Salinidad(psu)""]"
4,25408_52285_4030024_WIND_20250506172439_202605...,4030024,WIND,8760,8760,0,2025-05-06 00:00:00+00:00,2026-05-06 23:00:00+00:00,csv_columns,NaN,NaN,utf-8,\t,1,"[""Fecha (GMT)""]","{""wind_speed"": ""Velocidad del viento(m/s)"", ""w...",[],"[""Fecha (GMT)"", ""Velocidad del viento(m/s)"", ""..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
66,25410_52351_4058026_WAVE_20010101183202_202605...,4058026,WAVE,219494,219494,0,2001-01-01 00:00:00+00:00,2026-05-06 23:00:00+00:00,csv_columns,NaN,NaN,utf-8,\t,1,"[""Fecha (GMT)""]","{""hs"": ""Altura Signif. del Oleaje(m)"", ""tp"": ""...","[""Mar de viento: Direcc. Media de Proced.(0=N,...","[""Fecha (GMT)"", ""Altura Signif. del Oleaje(m)""..."
67,25410_52352_4058026_CURRENTS_20010101183202_20...,4058026,CURRENTS,39960,39960,0,2021-09-26 00:00:00+00:00,2026-05-05 23:00:00+00:00,csv_columns,NaN,NaN,utf-8,\t,1,"[""Fecha (GMT)""]","{""current_speed"": ""Velocidad media de Corrient...",[],"[""Fecha (GMT)"", ""Velocidad media de Corriente(..."
68,25410_52353_4058026_WATER_TEMP_20010101183202_...,4058026,WATER_TEMP,39960,39960,0,2021-09-26 00:00:00+00:00,2026-05-05 23:00:00+00:00,csv_columns,NaN,NaN,utf-8,\t,1,"[""Fecha (GMT)""]","{""sea_surface_temperature"": ""Temperatura del A...",[],"[""Fecha (GMT)"", ""Temperatura del Agua(ºC)""]"
69,25410_52354_4058026_SALINITY_20010101183202_20...,4058026,SALINITY,39960,39960,0,2021-09-26 00:00:00+00:00,2026-05-05 23:00:00+00:00,csv_columns,NaN,NaN,utf-8,\t,1,"[""Fecha (GMT)""]","{""sea_surface_salinity"": ""Salinidad(psu)""}",[],"[""Fecha (GMT)"", ""Salinidad(psu)""]"


In [51]:
print("Grupos cargados:")
print(standardized_by_group.keys())

print("\nErrores de lectura:")
display(read_errors_df)

if len(read_errors_df):
    print("\nErrores por tipo de archivo:")
    tmp = read_errors_df.copy()
    tmp["variable_group"] = tmp["filename"].str.extract(r"_(WAVE|WIND|CURRENTS|WATER_TEMP|SALINITY)_")
    display(tmp["variable_group"].value_counts(dropna=False).reset_index())

Grupos cargados:
dict_keys(['WAVE', 'CURRENTS', 'WATER_TEMP', 'SALINITY', 'WIND'])

Errores de lectura:


""


## Celda 9 — Unificar por grupos y validar coordenadas de puntos

In [52]:
group_frames = {}

for group, dfs in standardized_by_group.items():
    if dfs:
        group_frames[group] = pd.concat(dfs, ignore_index=True)
        print(group, group_frames[group].shape)

expected_groups = ["WAVE", "WIND", "CURRENTS", "WATER_TEMP", "SALINITY"]
missing_groups = [g for g in expected_groups if g not in group_frames]
print("Grupos ausentes:", missing_groups)


def decode_simar_canarias_point(point_id):
    """
    Reconstruye coordenadas aproximadas de nodos SIMAR/WANA de Canarias
    cuando los CSV descargados no incluyen lat/lon en cabecera.

    Para nodos 40xxxxx se usa la codificación regular de malla SIMAR
    compatible con nodos Canarias.

    Para nodos 10xxxxx se aplica codificación WANA Canarias.
    Estos puntos deben revisarse después con simar_point_to_zone.csv.
    """

    if pd.isna(point_id):
        return np.nan, np.nan, "missing_point_id"

    s = str(point_id).strip()

    if not re.match(r"^\d{7}$", s):
        return np.nan, np.nan, "cannot_decode"

    x = int(s[:4])
    y = int(s[4:])

    # Nodos SIMAR Canarias tipo 40xxxxx.
    # Ejemplo documentado: 4056013 ≈ lon -13.833, lat 28.333.
    if s.startswith("40"):
        lon = (x - 4222) / 12
        lat = (y + 327) / 12
        return float(lat), float(lon), "decoded_simar_40_grid"

    # Nodos WANA Canarias tipo 10xxxxx.
    # Se mantienen como fallback operativo y se validan por distancia a zona.
    if s.startswith("10"):
        lon = (x - 1222) / 12
        lat = (y + 321) / 12
        return float(lat), float(lon), "decoded_wana_10_grid"

    return np.nan, np.nan, "unknown_grid"


# Metadata de puntos SIMAR desde lo que haya podido extraerse.
point_meta_rows = []

for group, df_group in group_frames.items():
    tmp = (
        df_group[["simar_point_id", "point_lat", "point_lon"]]
        .dropna(subset=["simar_point_id"])
        .drop_duplicates()
        .copy()
    )
    tmp["variable_group"] = group
    point_meta_rows.append(tmp)

if not point_meta_rows:
    raise ValueError("No hay datos SIMAR estandarizados. Revisar errores de lectura.")

point_meta_raw = pd.concat(point_meta_rows, ignore_index=True)

# Resolver una coordenada por punto usando mediana de valores válidos.
point_meta = (
    point_meta_raw
    .groupby("simar_point_id", as_index=False)
    .agg(
        point_lat=("point_lat", "median"),
        point_lon=("point_lon", "median"),
        variable_groups=("variable_group", lambda x: ",".join(sorted(set(x)))),
    )
)

# Fallback: reconstruir coordenadas desde el identificador del nodo.
decoded_rows = point_meta["simar_point_id"].apply(decode_simar_canarias_point)

point_meta["decoded_lat"] = decoded_rows.apply(lambda x: x[0])
point_meta["decoded_lon"] = decoded_rows.apply(lambda x: x[1])
point_meta["coordinate_source"] = decoded_rows.apply(lambda x: x[2])

point_meta["point_lat"] = point_meta["point_lat"].fillna(point_meta["decoded_lat"])
point_meta["point_lon"] = point_meta["point_lon"].fillna(point_meta["decoded_lon"])

point_meta["coords_missing"] = point_meta["point_lat"].isna() | point_meta["point_lon"].isna()

point_meta["inside_bbox"] = (
    point_meta["point_lat"].between(BBOX_CANARIAS["lat_min"], BBOX_CANARIAS["lat_max"])
    & point_meta["point_lon"].between(BBOX_CANARIAS["lon_min"], BBOX_CANARIAS["lon_max"])
)

print("Puntos SIMAR únicos:", len(point_meta))
print("Puntos sin coordenadas:", int(point_meta["coords_missing"].sum()))
print("Puntos dentro bbox:", int(point_meta["inside_bbox"].sum()))

display(point_meta)

point_meta.to_csv(META_DIR / "simar_point_metadata_raw.csv", index=False)

if point_meta["coords_missing"].any():
    raise ValueError(
        "Hay puntos SIMAR sin coordenadas incluso tras aplicar el decodificador. "
        "Revisa simar_point_metadata_raw.csv."
    )

print("Coordenadas SIMAR reconstruidas correctamente.")

WAVE (2273996, 14)
CURRENTS (1203354, 10)
WATER_TEMP (1212066, 7)
SALINITY (1212066, 7)
WIND (2045718, 8)
Grupos ausentes: []
Puntos SIMAR únicos: 15
Puntos sin coordenadas: 0
Puntos dentro bbox: 13


,simar_point_id,point_lat,point_lon,variable_groups,decoded_lat,decoded_lon,coordinate_source,coords_missing,inside_bbox
0,1010012,27.750000,-17.666667,"CURRENTS,SALINITY,WATER_TEMP,WAVE",27.750000,-17.666667,decoded_wana_10_grid,False,True
1,1020014,27.916667,-16.833333,"CURRENTS,SALINITY,WATER_TEMP,WAVE,WIND",27.916667,-16.833333,decoded_wana_10_grid,False,True
2,4006016,28.583333,-18.000000,"CURRENTS,SALINITY,WATER_TEMP,WAVE,WIND",28.583333,-18.000000,decoded_simar_40_grid,False,True
3,4006024,29.250000,-18.000000,"SALINITY,WATER_TEMP,WAVE,WIND",29.250000,-18.000000,decoded_simar_40_grid,False,True
4,4016004,27.583333,-17.166667,"CURRENTS,SALINITY,WATER_TEMP,WAVE,WIND",27.583333,-17.166667,decoded_simar_40_grid,False,True
5,4022016,28.583333,-16.666667,"CURRENTS,SALINITY,WATER_TEMP,WAVE,WIND",28.583333,-16.666667,decoded_simar_40_grid,False,True
6,4024000,27.250000,-16.500000,"CURRENTS,SALINITY,WATER_TEMP,WAVE,WIND",27.250000,-16.500000,decoded_simar_40_grid,False,True
7,4024030,29.750000,-16.500000,"CURRENTS,SALINITY,WATER_TEMP,WAVE,WIND",29.750000,-16.500000,decoded_simar_40_grid,False,False
8,4030024,29.250000,-16.000000,"CURRENTS,SALINITY,WATER_TEMP,WAVE,WIND",29.250000,-16.000000,decoded_simar_40_grid,False,True
9,4038010,28.083333,-15.333333,"CURRENTS,SALINITY,WATER_TEMP,WAVE,WIND",28.083333,-15.333333,decoded_simar_40_grid,False,True


Coordenadas SIMAR reconstruidas correctamente.


## Celda 10 — Asignar cada punto SIMAR a la zona costera más cercana

In [53]:
gdf_points = gpd.GeoDataFrame(
    point_meta.copy(),
    geometry=gpd.points_from_xy(point_meta["point_lon"], point_meta["point_lat"]),
    crs="EPSG:4326",
)

gdf_points_m = gdf_points.to_crs("EPSG:3857")

nearest = gpd.sjoin_nearest(
    gdf_points_m,
    gdf_zones_m[["zona_id", "nombre_zona", "isla", "municipio", "geometry"]],
    how="left",
    distance_col="distance_to_zona_m",
)

nearest = (
    nearest
    .sort_values("distance_to_zona_m")
    .groupby("simar_point_id", as_index=False)
    .first()
)

point_zone = pd.DataFrame(
    nearest.drop(columns="geometry", errors="ignore")
)

point_zone["distance_to_zona_km"] = point_zone["distance_to_zona_m"] / 1000

point_zone_cols = [
    "simar_point_id",
    "point_lat",
    "point_lon",
    "variable_groups",
    "inside_bbox",
    "zona_id",
    "nombre_zona",
    "isla",
    "municipio",
    "distance_to_zona_km",
]

point_zone = point_zone[point_zone_cols].copy()

display(point_zone)

point_zone.to_csv(META_DIR / "simar_point_to_zone.csv", index=False)

print("Distancia punto SIMAR → zona más cercana, km:")
display(point_zone["distance_to_zona_km"].describe())

far_points = point_zone[point_zone["distance_to_zona_km"] > 75].copy()

if len(far_points):
    print("AVISO: hay puntos SIMAR a más de 75 km de la zona más cercana.")
    display(far_points)

,simar_point_id,point_lat,point_lon,variable_groups,inside_bbox,zona_id,nombre_zona,isla,municipio,distance_to_zona_km
0,1010012,27.750000,-17.666667,"CURRENTS,SALINITY,WATER_TEMP,WAVE",True,CAN_EH_PUERTO_DE_LA_ESTACA,Puerto de la Estaca,El Hierro,Valverde,26.812797
1,1020014,27.916667,-16.833333,"CURRENTS,SALINITY,WATER_TEMP,WAVE,WIND",True,CAN_TF_EL_CALLAO_0,El Callao,Tenerife,Arona,18.820321
2,4006016,28.583333,-18.000000,"CURRENTS,SALINITY,WATER_TEMP,WAVE,WIND",True,CAN_LP_EL_CHARCON,El Charcón,La Palma,Tazacorte,8.811155
3,4006024,29.250000,-18.000000,"SALINITY,WATER_TEMP,WAVE,WIND",True,CAN_LP_CALLEJONCITOS,Callejoncitos,La Palma,Garafía,55.498287
4,4016004,27.583333,-17.166667,"CURRENTS,SALINITY,WATER_TEMP,WAVE,WIND",True,CAN_LG_ERESES,Ereses,La Gomera,Alajeró,56.051708
5,4022016,28.583333,-16.666667,"CURRENTS,SALINITY,WATER_TEMP,WAVE,WIND",True,CAN_TF_LOS_ROQUES_1,Los Roques,Tenerife,San Juan de la Rambla,23.622819
6,4024000,27.250000,-16.500000,"CURRENTS,SALINITY,WATER_TEMP,WAVE,WIND",True,CAN_TF_AMARILLA,Amarilla,Tenerife,San Miguel de Abona,96.630836
7,4024030,29.750000,-16.500000,"CURRENTS,SALINITY,WATER_TEMP,WAVE,WIND",False,CAN_TF_ARENISCO,Arenisco,Tenerife,San Cristóbal de La Laguna,151.480270
8,4030024,29.250000,-16.000000,"CURRENTS,SALINITY,WATER_TEMP,WAVE,WIND",True,CAN_TF_ROQUE_BERMEJO,Roque Bermejo,Tenerife,Santa Cruz de Tenerife,86.518056
9,4038010,28.083333,-15.333333,"CURRENTS,SALINITY,WATER_TEMP,WAVE,WIND",True,CAN_GC_SAN_CRISTOBAL,San Cristóbal,Gran Canaria,Las Palmas de Gran Canaria,9.069595


Distancia punto SIMAR → zona más cercana, km:


,distance_to_zona_km
count,15.000000
mean,51.223204
std,44.539736
min,6.040665
25%,18.382968
50%,29.360374
75%,74.437814
max,151.480270


AVISO: hay puntos SIMAR a más de 75 km de la zona más cercana.


,simar_point_id,point_lat,point_lon,variable_groups,inside_bbox,zona_id,nombre_zona,isla,municipio,distance_to_zona_km
6,4024000,27.25,-16.5,"CURRENTS,SALINITY,WATER_TEMP,WAVE,WIND",True,CAN_TF_AMARILLA,Amarilla,Tenerife,San Miguel de Abona,96.630836
7,4024030,29.75,-16.5,"CURRENTS,SALINITY,WATER_TEMP,WAVE,WIND",False,CAN_TF_ARENISCO,Arenisco,Tenerife,San Cristóbal de La Laguna,151.480270
8,4030024,29.25,-16.0,"CURRENTS,SALINITY,WATER_TEMP,WAVE,WIND",True,CAN_TF_ROQUE_BERMEJO,Roque Bermejo,Tenerife,Santa Cruz de Tenerife,86.518056
10,4048030,29.75,-14.5,"CURRENTS,SALINITY,WATER_TEMP,WAVE,WIND",False,CAN_LZ_PLAYA_DEL_COCHINO,Playa del Cochino,Lanzarote,Yaiza,119.327987


## Celda 11 — Funciones de calidad, flags y gaps

In [54]:
VARIABLE_RANGES = {
    "hs": (0, 15),
    "hmax": (0, 25),
    "tp": (0, 35),
    "tm02": (0, 35),
    "wave_direction": (0, 360),
    "swell_height": (0, 15),
    "swell_period": (0, 35),
    "swell_direction": (0, 360),
    "wind_wave_height": (0, 15),
    "wind_wave_period": (0, 35),
    "wind_speed": (0, 60),
    "wind_direction": (0, 360),
    "wind_gust": (0, 80),
    "current_u": (-5, 5),
    "current_v": (-5, 5),
    "current_speed": (0, 5),
    "current_direction": (0, 360),
    "sea_surface_temperature": (10, 35),
    "sea_surface_salinity": (20, 45),
}


def add_quality_flags(df, variable_ranges):
    df = df.copy()

    for col, (vmin, vmax) in variable_ranges.items():
        if col not in df.columns:
            continue

        flag_col = f"{col}_flag"
        df[flag_col] = 0

        missing_mask = df[col].isna()
        outlier_mask = (~missing_mask) & ((df[col] < vmin) | (df[col] > vmax))

        df.loc[missing_mask, flag_col] = 1
        df.loc[outlier_mask, flag_col] = 2

        df[flag_col] = df[flag_col].astype("int8")

    return df


def attach_point_zone(df):
    df = df.copy()

    df = df.drop(columns=["point_lat", "point_lon"], errors="ignore")

    merged = df.merge(
        point_zone[
            [
                "simar_point_id",
                "point_lat",
                "point_lon",
                "zona_id",
                "nombre_zona",
                "isla",
                "municipio",
                "distance_to_zona_km",
            ]
        ],
        on="simar_point_id",
        how="left",
    )

    merged = merged.rename(columns={"point_lat": "lat", "point_lon": "lon"})
    merged["source"] = SOURCE_NAME
    merged["temporal_resolution"] = "hourly"
    merged["year"] = merged["timestamp"].dt.year.astype("Int64")

    return merged


def deduplicate_timeseries(df, subset_cols):
    before = len(df)
    df = df.sort_values(subset_cols).drop_duplicates(subset=subset_cols, keep="first").copy()
    return df, before - len(df)


def gap_report(df, name, point_col="simar_point_id", timestamp_col="timestamp", expected_hours=1):
    rows = []

    if df.empty:
        return pd.DataFrame(
            columns=[
                "table",
                "simar_point_id",
                "timestamp_min",
                "timestamp_max",
                "rows",
                "gaps_count",
                "max_gap_hours",
                "expected_hours",
            ]
        )

    for point_id, g in df[[point_col, timestamp_col]].dropna().groupby(point_col):
        ts = g[timestamp_col].sort_values().drop_duplicates()
        diffs = ts.diff().dropna()
        diffs_h = diffs.dt.total_seconds() / 3600

        gaps = diffs_h[diffs_h > expected_hours * 1.5]

        rows.append(
            {
                "table": name,
                "simar_point_id": point_id,
                "timestamp_min": ts.min(),
                "timestamp_max": ts.max(),
                "rows": len(ts),
                "gaps_count": int(len(gaps)),
                "max_gap_hours": float(gaps.max()) if len(gaps) else 0.0,
                "expected_hours": expected_hours,
            }
        )

    return pd.DataFrame(rows)


def quality_summary(df, table_name):
    rows = [
        {"table": table_name, "metric": "rows", "value": len(df)},
        {"table": table_name, "metric": "unique_simar_points", "value": df["simar_point_id"].nunique() if "simar_point_id" in df.columns else 0},
        {"table": table_name, "metric": "unique_zona_id", "value": df["zona_id"].nunique() if "zona_id" in df.columns else 0},
        {"table": table_name, "metric": "timestamp_min", "value": df["timestamp"].min() if "timestamp" in df.columns and len(df) else pd.NaT},
        {"table": table_name, "metric": "timestamp_max", "value": df["timestamp"].max() if "timestamp" in df.columns and len(df) else pd.NaT},
        {"table": table_name, "metric": "missing_zona_id", "value": int(df["zona_id"].isna().sum()) if "zona_id" in df.columns else np.nan},
        {"table": table_name, "metric": "missing_lat", "value": int(df["lat"].isna().sum()) if "lat" in df.columns else np.nan},
        {"table": table_name, "metric": "missing_lon", "value": int(df["lon"].isna().sum()) if "lon" in df.columns else np.nan},
    ]

    for col in df.columns:
        if col.endswith("_flag"):
            rows.append(
                {
                    "table": table_name,
                    "metric": f"{col}_missing_pct",
                    "value": float((df[col] == 1).mean() * 100) if len(df) else 0.0,
                }
            )
            rows.append(
                {
                    "table": table_name,
                    "metric": f"{col}_outlier_pct",
                    "value": float((df[col] == 2).mean() * 100) if len(df) else 0.0,
                }
            )

    return pd.DataFrame(rows)

## Celda 12 — Construir `silver/ocean_hourly/` desde SIMAR WAVE

In [55]:
ocean_columns = [
    "timestamp",
    "zona_id",
    "simar_point_id",
    "lat",
    "lon",
    "source",
    "hs",
    "hmax",
    "tp",
    "tm02",
    "wave_direction",
    "swell_height",
    "swell_period",
    "swell_direction",
    "wind_wave_height",
    "wind_wave_period",
    "stokes_drift",
    "distance_to_zona_km",
    "temporal_resolution",
    "year",
    "isla",
]

if "WAVE" in group_frames:
    ocean_hourly = group_frames["WAVE"].copy()

    # Stokes drift no está normalmente en SIMAR.
    if "stokes_drift" not in ocean_hourly.columns:
        ocean_hourly["stokes_drift"] = np.nan

    ocean_hourly = attach_point_zone(ocean_hourly)

    for col in ocean_columns:
        if col not in ocean_hourly.columns:
            ocean_hourly[col] = np.nan

    ocean_hourly = ocean_hourly[ocean_columns].copy()

    ocean_hourly, ocean_duplicates = deduplicate_timeseries(
        ocean_hourly,
        subset_cols=["simar_point_id", "timestamp", "source"],
    )

    ocean_hourly = add_quality_flags(ocean_hourly, VARIABLE_RANGES)

else:
    ocean_hourly = pd.DataFrame(columns=ocean_columns)
    ocean_duplicates = 0

print("ocean_hourly shape:", ocean_hourly.shape)
print("duplicados eliminados:", ocean_duplicates)
display(ocean_hourly.head())

ocean_hourly shape: (2273996, 31)
duplicados eliminados: 0


,timestamp,zona_id,simar_point_id,lat,lon,source,hs,hmax,tp,tm02,...,hs_flag,hmax_flag,tp_flag,tm02_flag,wave_direction_flag,swell_height_flag,swell_period_flag,swell_direction_flag,wind_wave_height_flag,wind_wave_period_flag
17520,2025-05-06 00:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,1010012,27.75,-17.666667,SIMAR,0.97,NaN,9.10,4.07,...,0,1,0,0,0,0,0,0,0,1
17521,2025-05-06 01:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,1010012,27.75,-17.666667,SIMAR,0.94,NaN,10.01,4.19,...,0,1,0,0,0,0,0,0,0,1
17522,2025-05-06 02:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,1010012,27.75,-17.666667,SIMAR,0.91,NaN,10.01,4.48,...,0,1,0,0,0,0,0,0,0,1
17523,2025-05-06 03:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,1010012,27.75,-17.666667,SIMAR,0.90,NaN,12.11,4.78,...,0,1,0,0,0,0,0,0,0,1
17524,2025-05-06 04:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,1010012,27.75,-17.666667,SIMAR,0.90,NaN,12.11,4.94,...,0,1,0,0,0,0,0,0,0,1


## Celda 13 — Construir `silver/meteo_hourly/` desde SIMAR WIND

In [56]:
meteo_columns = [
    "timestamp",
    "station_id",
    "zona_id",
    "simar_point_id",
    "lat",
    "lon",
    "source",
    "wind_speed",
    "wind_direction",
    "wind_gust",
    "temperature_air",
    "pressure",
    "precipitation",
    "humidity",
    "u10",
    "v10",
    "distance_to_zona_km",
    "temporal_resolution",
    "year",
    "isla",
]

if "WIND" in group_frames:
    meteo_hourly = group_frames["WIND"].copy()
    meteo_hourly = attach_point_zone(meteo_hourly)

    meteo_hourly["station_id"] = "SIMAR_" + meteo_hourly["simar_point_id"].astype(str)

    # Calcular componentes u/v desde velocidad/dirección si existen.
    # Convención meteorológica: dirección desde donde sopla el viento.
    if {"wind_speed", "wind_direction"}.issubset(meteo_hourly.columns):
        wd_rad = np.deg2rad(meteo_hourly["wind_direction"])
        meteo_hourly["u10"] = -meteo_hourly["wind_speed"] * np.sin(wd_rad)
        meteo_hourly["v10"] = -meteo_hourly["wind_speed"] * np.cos(wd_rad)

    for col in meteo_columns:
        if col not in meteo_hourly.columns:
            meteo_hourly[col] = np.nan

    meteo_hourly = meteo_hourly[meteo_columns].copy()

    meteo_hourly, meteo_duplicates = deduplicate_timeseries(
        meteo_hourly,
        subset_cols=["station_id", "timestamp", "source"],
    )

    meteo_hourly = add_quality_flags(meteo_hourly, VARIABLE_RANGES)

else:
    meteo_hourly = pd.DataFrame(columns=meteo_columns)
    meteo_duplicates = 0

print("meteo_hourly shape:", meteo_hourly.shape)
print("duplicados eliminados:", meteo_duplicates)
display(meteo_hourly.head())

meteo_hourly shape: (2045718, 23)
duplicados eliminados: 0


,timestamp,station_id,zona_id,simar_point_id,lat,lon,source,wind_speed,wind_direction,wind_gust,...,humidity,u10,v10,distance_to_zona_km,temporal_resolution,year,isla,wind_speed_flag,wind_direction_flag,wind_gust_flag
8760,2025-05-06 00:00:00+00:00,SIMAR_1020014,CAN_TF_EL_CALLAO_0,1020014,27.916667,-16.833333,SIMAR,5.87,59.0,NaN,...,NaN,-5.031572,-3.023273,18.820321,hourly,2025,Tenerife,0,0,1
8761,2025-05-06 01:00:00+00:00,SIMAR_1020014,CAN_TF_EL_CALLAO_0,1020014,27.916667,-16.833333,SIMAR,5.28,60.0,NaN,...,NaN,-4.572614,-2.640000,18.820321,hourly,2025,Tenerife,0,0,1
8762,2025-05-06 02:00:00+00:00,SIMAR_1020014,CAN_TF_EL_CALLAO_0,1020014,27.916667,-16.833333,SIMAR,5.36,62.0,NaN,...,NaN,-4.732599,-2.516368,18.820321,hourly,2025,Tenerife,0,0,1
8763,2025-05-06 03:00:00+00:00,SIMAR_1020014,CAN_TF_EL_CALLAO_0,1020014,27.916667,-16.833333,SIMAR,5.17,54.0,NaN,...,NaN,-4.182618,-3.038850,18.820321,hourly,2025,Tenerife,0,0,1
8764,2025-05-06 04:00:00+00:00,SIMAR_1020014,CAN_TF_EL_CALLAO_0,1020014,27.916667,-16.833333,SIMAR,5.51,51.0,NaN,...,NaN,-4.282074,-3.467555,18.820321,hourly,2025,Tenerife,0,0,1


## Celda 14 — Construir `silver/ocean_physics/` desde CURRENTS / WATER_TEMP / SALINITY

In [57]:
physics_base_cols = [
    "timestamp",
    "simar_point_id",
]

physics_value_cols = [
    "current_u",
    "current_v",
    "current_speed",
    "current_direction",
    "sea_surface_temperature",
    "sea_surface_salinity",
]

physics_parts = []

for group in ["CURRENTS", "WATER_TEMP", "SALINITY"]:
    if group not in group_frames:
        continue

    df_group = group_frames[group].copy()

    keep_cols = physics_base_cols + [c for c in physics_value_cols if c in df_group.columns]
    df_group = df_group[keep_cols].copy()

    # Deduplicar dentro de cada grupo antes de merge.
    df_group = (
        df_group
        .sort_values(["simar_point_id", "timestamp"])
        .drop_duplicates(subset=["simar_point_id", "timestamp"], keep="first")
    )

    physics_parts.append(df_group)

if physics_parts:
    ocean_physics = reduce(
        lambda left, right: pd.merge(
            left,
            right,
            on=["simar_point_id", "timestamp"],
            how="outer",
        ),
        physics_parts,
    )

    ocean_physics = attach_point_zone(ocean_physics)

    physics_columns = [
        "timestamp",
        "zona_id",
        "simar_point_id",
        "lat",
        "lon",
        "source",
        "current_u",
        "current_v",
        "current_speed",
        "current_direction",
        "sea_surface_temperature",
        "sea_surface_salinity",
        "distance_to_zona_km",
        "temporal_resolution",
        "year",
        "isla",
    ]

    for col in physics_columns:
        if col not in ocean_physics.columns:
            ocean_physics[col] = np.nan

    ocean_physics = ocean_physics[physics_columns].copy()

    ocean_physics, physics_duplicates = deduplicate_timeseries(
        ocean_physics,
        subset_cols=["simar_point_id", "timestamp", "source"],
    )

    ocean_physics = add_quality_flags(ocean_physics, VARIABLE_RANGES)

else:
    ocean_physics = pd.DataFrame(
        columns=[
            "timestamp",
            "zona_id",
            "simar_point_id",
            "lat",
            "lon",
            "source",
            "current_u",
            "current_v",
            "current_speed",
            "current_direction",
            "sea_surface_temperature",
            "sea_surface_salinity",
            "distance_to_zona_km",
            "temporal_resolution",
            "year",
            "isla",
        ]
    )
    physics_duplicates = 0

print("ocean_physics shape:", ocean_physics.shape)
print("duplicados eliminados:", physics_duplicates)
display(ocean_physics.head())

ocean_physics shape: (1212066, 22)
duplicados eliminados: 0


,timestamp,zona_id,simar_point_id,lat,lon,source,current_u,current_v,current_speed,current_direction,...,distance_to_zona_km,temporal_resolution,year,isla,current_u_flag,current_v_flag,current_speed_flag,current_direction_flag,sea_surface_temperature_flag,sea_surface_salinity_flag
0,2025-05-06 00:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,1010012,27.75,-17.666667,SIMAR,0.157849,0.266802,0.310,30.61,...,26.812797,hourly,2025,El Hierro,0,0,0,0,0,0
1,2025-05-06 01:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,1010012,27.75,-17.666667,SIMAR,0.130704,0.238538,0.272,28.72,...,26.812797,hourly,2025,El Hierro,0,0,0,0,0,0
2,2025-05-06 02:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,1010012,27.75,-17.666667,SIMAR,0.097896,0.227860,0.248,23.25,...,26.812797,hourly,2025,El Hierro,0,0,0,0,0,0
3,2025-05-06 03:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,1010012,27.75,-17.666667,SIMAR,0.064995,0.229993,0.239,15.78,...,26.812797,hourly,2025,El Hierro,0,0,0,0,0,0
4,2025-05-06 04:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,1010012,27.75,-17.666667,SIMAR,0.035895,0.239323,0.242,8.53,...,26.812797,hourly,2025,El Hierro,0,0,0,0,0,0


## Celda 15 — Reportes de calidad y gaps

In [58]:
ocean_gap_report = gap_report(ocean_hourly, "ocean_hourly")
meteo_gap_report = gap_report(meteo_hourly, "meteo_hourly")
physics_gap_report = gap_report(ocean_physics, "ocean_physics")

ocean_quality = quality_summary(ocean_hourly, "ocean_hourly")
meteo_quality = quality_summary(meteo_hourly, "meteo_hourly")
physics_quality = quality_summary(ocean_physics, "ocean_physics")

duplicates_summary = pd.DataFrame(
    [
        {"table": "ocean_hourly", "duplicates_removed": ocean_duplicates},
        {"table": "meteo_hourly", "duplicates_removed": meteo_duplicates},
        {"table": "ocean_physics", "duplicates_removed": physics_duplicates},
    ]
)

quality_all = pd.concat(
    [
        ocean_quality,
        meteo_quality,
        physics_quality,
    ],
    ignore_index=True,
)

gaps_all = pd.concat(
    [
        ocean_gap_report,
        meteo_gap_report,
        physics_gap_report,
    ],
    ignore_index=True,
)

print("Resumen de calidad:")
display(quality_all)

print("Resumen de duplicados:")
display(duplicates_summary)

print("Gaps:")
display(gaps_all)

quality_all.to_csv(QC_DIR / "quality_simar_tables_summary.csv", index=False)
gaps_all.to_csv(QC_DIR / "quality_simar_gaps_by_point.csv", index=False)
duplicates_summary.to_csv(QC_DIR / "quality_simar_duplicates_summary.csv", index=False)
point_zone.to_csv(META_DIR / "simar_point_to_zone.csv", index=False)

Resumen de calidad:


,table,metric,value
0,ocean_hourly,rows,2273996
1,ocean_hourly,unique_simar_points,14
2,ocean_hourly,unique_zona_id,14
3,ocean_hourly,timestamp_min,2000-01-01 00:00:00+00:00
4,ocean_hourly,timestamp_max,2026-05-06 23:00:00+00:00
...,...,...,...
57,ocean_physics,current_direction_flag_outlier_pct,0.0
58,ocean_physics,sea_surface_temperature_flag_missing_pct,0.0
59,ocean_physics,sea_surface_temperature_flag_outlier_pct,0.00429
60,ocean_physics,sea_surface_salinity_flag_missing_pct,0.0


Resumen de duplicados:


,table,duplicates_removed
0,ocean_hourly,0
1,meteo_hourly,0
2,ocean_physics,0


Gaps:


,table,simar_point_id,timestamp_min,timestamp_max,rows,gaps_count,max_gap_hours,expected_hours
0,ocean_hourly,1010012,2025-05-06 00:00:00+00:00,2026-05-06 23:00:00+00:00,8760,2,13.0,1
1,ocean_hourly,1020014,2025-05-06 00:00:00+00:00,2026-05-06 23:00:00+00:00,8760,2,13.0,1
2,ocean_hourly,4006016,2001-01-01 00:00:00+00:00,2026-05-06 23:00:00+00:00,219494,267,567.0,1
3,ocean_hourly,4006024,2025-05-06 00:00:00+00:00,2026-05-06 23:00:00+00:00,8760,2,13.0,1
4,ocean_hourly,4016004,2001-01-01 00:00:00+00:00,2026-05-06 23:00:00+00:00,219494,267,567.0,1
5,ocean_hourly,4022016,2001-01-01 00:00:00+00:00,2026-05-06 23:00:00+00:00,219494,267,567.0,1
6,ocean_hourly,4024000,2000-01-01 00:00:00+00:00,2026-05-06 23:00:00+00:00,228302,263,567.0,1
7,ocean_hourly,4024030,2000-01-01 00:00:00+00:00,2026-05-06 23:00:00+00:00,228314,267,531.0,1
8,ocean_hourly,4030024,2025-05-06 00:00:00+00:00,2026-05-06 23:00:00+00:00,8760,2,13.0,1
9,ocean_hourly,4038010,2000-01-01 00:00:00+00:00,2026-05-06 23:00:00+00:00,228278,267,567.0,1


## Celda 16 — Validaciones finales antes de guardar

In [59]:
def variable_missing_pct(df, col):
    if col not in df.columns or df.empty:
        return 100.0
    return float(df[col].isna().mean() * 100)


variable_checks = pd.DataFrame(
    [
        {
            "table": "ocean_hourly",
            "variable": "hs",
            "missing_pct": variable_missing_pct(ocean_hourly, "hs"),
            "max_allowed_missing_pct": 20,
        },
        {
            "table": "ocean_hourly",
            "variable": "tp",
            "missing_pct": variable_missing_pct(ocean_hourly, "tp"),
            "max_allowed_missing_pct": 20,
        },
        {
            "table": "meteo_hourly",
            "variable": "wind_speed",
            "missing_pct": variable_missing_pct(meteo_hourly, "wind_speed"),
            "max_allowed_missing_pct": 20,
        },
        {
            "table": "meteo_hourly",
            "variable": "wind_direction",
            "missing_pct": variable_missing_pct(meteo_hourly, "wind_direction"),
            "max_allowed_missing_pct": 20,
        },
        {
            "table": "meteo_hourly",
            "variable": "u10",
            "missing_pct": variable_missing_pct(meteo_hourly, "u10"),
            "max_allowed_missing_pct": 20,
        },
        {
            "table": "meteo_hourly",
            "variable": "v10",
            "missing_pct": variable_missing_pct(meteo_hourly, "v10"),
            "max_allowed_missing_pct": 20,
        },
        {
            "table": "ocean_physics",
            "variable": "current_speed",
            "missing_pct": variable_missing_pct(ocean_physics, "current_speed"),
            "max_allowed_missing_pct": 20,
        },
        {
            "table": "ocean_physics",
            "variable": "current_direction",
            "missing_pct": variable_missing_pct(ocean_physics, "current_direction"),
            "max_allowed_missing_pct": 20,
        },
        {
            "table": "ocean_physics",
            "variable": "sea_surface_temperature",
            "missing_pct": variable_missing_pct(ocean_physics, "sea_surface_temperature"),
            "max_allowed_missing_pct": 20,
        },
        {
            "table": "ocean_physics",
            "variable": "sea_surface_salinity",
            "missing_pct": variable_missing_pct(ocean_physics, "sea_surface_salinity"),
            "max_allowed_missing_pct": 20,
        },
    ]
)

display(variable_checks)

bad_variables = variable_checks[
    variable_checks["missing_pct"] > variable_checks["max_allowed_missing_pct"]
]

if len(bad_variables):
    raise ValueError(
        "Hay variables clave con demasiados nulos. Revisar mapeo antes de guardar."
    )

variable_checks.to_csv(
    QC_DIR / "quality_simar_key_variable_missing_checks.csv",
    index=False,
)

print("Validación de variables clave superada.")

,table,variable,missing_pct,max_allowed_missing_pct
0,ocean_hourly,hs,0.000000,20
1,ocean_hourly,tp,0.000000,20
2,meteo_hourly,wind_speed,0.000000,20
3,meteo_hourly,wind_direction,0.000000,20
4,meteo_hourly,u10,0.000000,20
5,meteo_hourly,v10,0.000000,20
6,ocean_physics,current_speed,0.718773,20
7,ocean_physics,current_direction,0.718773,20
8,ocean_physics,sea_surface_temperature,0.000000,20
9,ocean_physics,sea_surface_salinity,0.000000,20


Validación de variables clave superada.


In [60]:
def validate_before_save(df, table_name, required_columns, allow_empty=False):
    print(f"Validando {table_name}...")

    if df.empty and not allow_empty:
        raise ValueError(f"{table_name} está vacío. No se guarda.")

    missing_cols = [c for c in required_columns if c not in df.columns]
    if missing_cols:
        raise ValueError(f"{table_name} no tiene columnas requeridas: {missing_cols}")

    if not df.empty:
        if df["timestamp"].isna().any():
            raise ValueError(f"{table_name} tiene timestamps nulos.")

        ts_min = df["timestamp"].min()
        ts_max = df["timestamp"].max()

        print("  timestamp_min:", ts_min)
        print("  timestamp_max:", ts_max)

        if ts_min < MIN_VALID_TS or ts_max > MAX_VALID_TS:
            raise ValueError(
                f"{table_name} tiene timestamps fuera de rango real "
                f"({ts_min} - {ts_max}). No guardar."
            )

        if df["zona_id"].isna().any():
            raise ValueError(f"{table_name} tiene zona_id nulos.")

        if df["lat"].isna().any() or df["lon"].isna().any():
            raise ValueError(f"{table_name} tiene coordenadas nulas.")

        if df["year"].isna().any():
            raise ValueError(f"{table_name} tiene year nulo.")

        invalid_years = ~df["year"].between(MIN_VALID_TS.year, MAX_VALID_TS.year)
        if invalid_years.any():
            raise ValueError(
                f"{table_name} tiene años inválidos: "
                f"{sorted(df.loc[invalid_years, 'year'].dropna().unique().tolist())}"
            )

        if df["isla"].isna().any():
            raise ValueError(f"{table_name} tiene isla nula.")

        outside_bbox = (
            ~df["lat"].between(BBOX_CANARIAS["lat_min"], BBOX_CANARIAS["lat_max"])
            | ~df["lon"].between(BBOX_CANARIAS["lon_min"], BBOX_CANARIAS["lon_max"])
        ).sum()

        if outside_bbox > 0:
            print(
                f"  AVISO: {table_name} tiene {outside_bbox} filas fuera del bbox. "
                "Si son puntos SIMAR offshore cercanos, puede ser aceptable; revisa point_zone."
            )

    print(f"{table_name}: OK")


def validate_duplicate_loss(raw_group_frames, final_tables):
    checks = []

    if "WAVE" in raw_group_frames:
        checks.append(("ocean_hourly", len(raw_group_frames["WAVE"]), len(final_tables["ocean_hourly"])))

    if "WIND" in raw_group_frames:
        checks.append(("meteo_hourly", len(raw_group_frames["WIND"]), len(final_tables["meteo_hourly"])))

    # En ocean_physics hay merge entre 3 fuentes; no se compara igual.

    for table_name, raw_rows, final_rows in checks:
        if raw_rows == 0:
            continue

        retained_pct = final_rows / raw_rows * 100
        print(f"{table_name}: filas retenidas {retained_pct:.2f}% ({final_rows}/{raw_rows})")

        if retained_pct < 80:
            raise ValueError(
                f"{table_name} perdió demasiadas filas tras deduplicar. "
                "Esto suele indicar timestamps mal parseados. No guardar."
            )


validate_before_save(
    ocean_hourly,
    "ocean_hourly",
    required_columns=["timestamp", "zona_id", "simar_point_id", "lat", "lon", "source", "year", "isla"],
    allow_empty=False,
)

validate_before_save(
    meteo_hourly,
    "meteo_hourly",
    required_columns=["timestamp", "zona_id", "station_id", "lat", "lon", "source", "year", "isla"],
    allow_empty=True,
)

validate_before_save(
    ocean_physics,
    "ocean_physics",
    required_columns=["timestamp", "zona_id", "simar_point_id", "lat", "lon", "source", "year", "isla"],
    allow_empty=True,
)

validate_duplicate_loss(
    group_frames,
    {
        "ocean_hourly": ocean_hourly,
        "meteo_hourly": meteo_hourly,
    },
)

print("Validaciones finales superadas.")

Validando ocean_hourly...
  timestamp_min: 2000-01-01 00:00:00+00:00
  timestamp_max: 2026-05-06 23:00:00+00:00
  AVISO: ocean_hourly tiene 456628 filas fuera del bbox. Si son puntos SIMAR offshore cercanos, puede ser aceptable; revisa point_zone.
ocean_hourly: OK
Validando meteo_hourly...
  timestamp_min: 2000-01-01 00:00:00+00:00
  timestamp_max: 2026-05-06 23:00:00+00:00
  AVISO: meteo_hourly tiene 456628 filas fuera del bbox. Si son puntos SIMAR offshore cercanos, puede ser aceptable; revisa point_zone.
meteo_hourly: OK
Validando ocean_physics...
  timestamp_min: 2011-03-31 00:00:00+00:00
  timestamp_max: 2026-05-05 23:00:00+00:00
  AVISO: ocean_physics tiene 261678 filas fuera del bbox. Si son puntos SIMAR offshore cercanos, puede ser aceptable; revisa point_zone.
ocean_physics: OK
ocean_hourly: filas retenidas 100.00% (2273996/2273996)
meteo_hourly: filas retenidas 100.00% (2045718/2045718)
Validaciones finales superadas.


## Celda 17 — Guardar Parquet particionado

In [61]:
import pyarrow as pa
import pyarrow.parquet as pq


def remove_existing_source_partition(base_dir, source_name=SOURCE_NAME):
    source_path = base_dir / f"source={source_name}"
    if source_path.exists():
        shutil.rmtree(source_path)
        print("Eliminada partición antigua:", source_path)


def write_partitioned_parquet(df, base_dir, table_name):
    if df.empty:
        print(f"{table_name}: DataFrame vacío. No se guarda.")
        return

    df = df.copy()

    df["source"] = df["source"].fillna(SOURCE_NAME).astype(str)
    df["isla"] = df["isla"].fillna("ISLA_DESCONOCIDA").astype(str)
    df["year"] = df["year"].astype("int64")

    table = pa.Table.from_pandas(df, preserve_index=False)

    pq.write_to_dataset(
        table,
        root_path=str(base_dir),
        partition_cols=["source", "year", "isla"],
        compression="snappy",
    )

    print(f"{table_name} guardado en:", base_dir)


# Evitar mezclar una ejecución antigua de SIMAR con la nueva.
for base_dir in [OUT_OCEAN_DIR, OUT_METEO_DIR, OUT_PHYSICS_DIR]:
    remove_existing_source_partition(base_dir, SOURCE_NAME)

write_partitioned_parquet(ocean_hourly, OUT_OCEAN_DIR, "ocean_hourly")
write_partitioned_parquet(meteo_hourly, OUT_METEO_DIR, "meteo_hourly")
write_partitioned_parquet(ocean_physics, OUT_PHYSICS_DIR, "ocean_physics")

print("Guardado finalizado.")

Eliminada partición antigua: /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/ocean_hourly/source=SIMAR
Eliminada partición antigua: /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/meteo_hourly/source=SIMAR
Eliminada partición antigua: /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/ocean_physics/source=SIMAR
ocean_hourly guardado en: /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/ocean_hourly
meteo_hourly guardado en: /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/meteo_hourly
ocean_physics guardado en: /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/ocean_physics
Guardado finalizado.


## Celda 18 — Comprobación final de lectura

In [62]:
def read_dataset_if_exists(path):
    if not path.exists():
        print("No existe:", path)
        return pd.DataFrame()

    try:
        return pd.read_parquet(path)
    except Exception as e:
        print("No se pudo leer:", path, e)
        return pd.DataFrame()


test_ocean = read_dataset_if_exists(OUT_OCEAN_DIR)
test_meteo = read_dataset_if_exists(OUT_METEO_DIR)
test_physics = read_dataset_if_exists(OUT_PHYSICS_DIR)

print("test_ocean:", test_ocean.shape)
print("test_meteo:", test_meteo.shape)
print("test_physics:", test_physics.shape)

if len(test_ocean):
    display(test_ocean.head())

if len(test_meteo):
    display(test_meteo.head())

if len(test_physics):
    display(test_physics.head())

print("Archivos de calidad generados:")
for p in sorted(QC_DIR.glob("quality_simar*.csv")):
    print("-", p)

print("Metadatos generados:")
for p in sorted(META_DIR.glob("simar*.csv")):
    print("-", p)

test_ocean: (2273996, 31)
test_meteo: (2045718, 23)
test_physics: (1212066, 22)


,timestamp,zona_id,simar_point_id,lat,lon,hs,hmax,tp,tm02,wave_direction,...,tm02_flag,wave_direction_flag,swell_height_flag,swell_period_flag,swell_direction_flag,wind_wave_height_flag,wind_wave_period_flag,source,year,isla
0,2000-01-01 00:00:00+00:00,CAN_FV_GRAN_TARAJAL,4054011,28.166667,-14.0,0.67,NaN,5.08,3.54,66,...,0,0,0,2,0,0,1,SIMAR,2000,Fuerteventura
1,2000-01-01 01:00:00+00:00,CAN_FV_GRAN_TARAJAL,4054011,28.166667,-14.0,0.66,NaN,4.98,3.50,65,...,0,0,0,2,0,0,1,SIMAR,2000,Fuerteventura
2,2000-01-01 02:00:00+00:00,CAN_FV_GRAN_TARAJAL,4054011,28.166667,-14.0,0.66,NaN,4.93,3.48,65,...,0,0,0,2,0,0,1,SIMAR,2000,Fuerteventura
3,2000-01-01 03:00:00+00:00,CAN_FV_GRAN_TARAJAL,4054011,28.166667,-14.0,0.67,NaN,4.95,3.49,64,...,0,0,0,2,0,0,1,SIMAR,2000,Fuerteventura
4,2000-01-01 04:00:00+00:00,CAN_FV_GRAN_TARAJAL,4054011,28.166667,-14.0,0.69,NaN,5.00,3.51,64,...,0,0,0,2,0,0,1,SIMAR,2000,Fuerteventura


,timestamp,station_id,zona_id,simar_point_id,lat,lon,wind_speed,wind_direction,wind_gust,temperature_air,...,u10,v10,distance_to_zona_km,temporal_resolution,wind_speed_flag,wind_direction_flag,wind_gust_flag,source,year,isla
0,2000-01-01 00:00:00+00:00,SIMAR_4038010,CAN_GC_SAN_CRISTOBAL,4038010,28.083333,-15.333333,8.4,42.0,NaN,NaN,...,-5.620697,-6.242417,9.069595,hourly,0,0,1,SIMAR,2000,Gran Canaria
1,2000-01-01 01:00:00+00:00,SIMAR_4038010,CAN_GC_SAN_CRISTOBAL,4038010,28.083333,-15.333333,8.6,43.0,NaN,NaN,...,-5.865186,-6.289642,9.069595,hourly,0,0,1,SIMAR,2000,Gran Canaria
2,2000-01-01 02:00:00+00:00,SIMAR_4038010,CAN_GC_SAN_CRISTOBAL,4038010,28.083333,-15.333333,8.6,45.0,NaN,NaN,...,-6.081118,-6.081118,9.069595,hourly,0,0,1,SIMAR,2000,Gran Canaria
3,2000-01-01 03:00:00+00:00,SIMAR_4038010,CAN_GC_SAN_CRISTOBAL,4038010,28.083333,-15.333333,8.5,46.0,NaN,NaN,...,-6.114388,-5.904596,9.069595,hourly,0,0,1,SIMAR,2000,Gran Canaria
4,2000-01-01 04:00:00+00:00,SIMAR_4038010,CAN_GC_SAN_CRISTOBAL,4038010,28.083333,-15.333333,8.3,48.0,NaN,NaN,...,-6.168102,-5.553784,9.069595,hourly,0,0,1,SIMAR,2000,Gran Canaria


,timestamp,zona_id,simar_point_id,lat,lon,current_u,current_v,current_speed,current_direction,sea_surface_temperature,...,temporal_resolution,current_u_flag,current_v_flag,current_speed_flag,current_direction_flag,sea_surface_temperature_flag,sea_surface_salinity_flag,source,year,isla
0,2011-03-31 00:00:00+00:00,CAN_FV_PLAYA_DEL_VALLE,4050016,28.583333,-14.333333,-0.035898,-0.149758,0.154,193.48,18.639,...,hourly,0,0,0,0,0,0,SIMAR,2011,Fuerteventura
1,2011-03-31 01:00:00+00:00,CAN_FV_PLAYA_DEL_VALLE,4050016,28.583333,-14.333333,-0.051495,-0.158862,0.167,197.96,18.638,...,hourly,0,0,0,0,0,0,SIMAR,2011,Fuerteventura
2,2011-03-31 02:00:00+00:00,CAN_FV_PLAYA_DEL_VALLE,4050016,28.583333,-14.333333,-0.064448,-0.168067,0.180,200.98,18.637,...,hourly,0,0,0,0,0,0,SIMAR,2011,Fuerteventura
3,2011-03-31 03:00:00+00:00,CAN_FV_PLAYA_DEL_VALLE,4050016,28.583333,-14.333333,-0.072649,-0.175562,0.190,202.48,18.632,...,hourly,0,0,0,0,0,0,SIMAR,2011,Fuerteventura
4,2011-03-31 04:00:00+00:00,CAN_FV_PLAYA_DEL_VALLE,4050016,28.583333,-14.333333,-0.078461,-0.175237,0.192,204.12,18.630,...,hourly,0,0,0,0,0,0,SIMAR,2011,Fuerteventura


Archivos de calidad generados:
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/_quality_reports/quality_simar_duplicates_summary.csv
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/_quality_reports/quality_simar_file_summary.csv
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/_quality_reports/quality_simar_gaps_by_point.csv
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/_quality_reports/quality_simar_key_variable_missing_checks.csv
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/_quality_reports/quality_simar_read_errors.csv
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/_quality_reports/quality_simar_tables_summary.csv
Metadatos generados:
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/_metadata/simar_point_metadata_raw.csv
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/_metadata/simar_point_to_zone.csv
